# Viento solar — análisis reproducible

Cuaderno autocontenido: escribe los módulos del proyecto en disco y los usa.
No hace falta clonar el repositorio ni tener credenciales.

**Para variar la longitud de ventana, cambia `WINDOW_SIZE` en la celda 5.**
Cada punto son 4 minutos, así que `WINDOW_SIZE = 360` equivale a 24 horas.


## 1. Dependencias

In [ ]:
!pip install -q cdflib
print('listo')

## 2. Datos

Monta el Drive donde están los CDF y los catálogos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/tesis/data'   # ajusta si tu ruta es otra

import os
for f in sorted(os.listdir(DATA)):
    print(f)

## 3. El código del proyecto

Esta celda escribe los módulos tal como están en el repositorio. Es el mismo
código, sin modificar.

In [ ]:
# Los modulos van en base64 para que sus comillas no rompan nada.
import base64, pathlib

ARCHIVOS = {
    "tesis/__init__.py": "",
    "tesis/data/__init__.py": "",
    "tesis/features/__init__.py": "",
    "tesis/models/__init__.py": "",
    "tesis/data/cdf.py": "IiIiQ2FyZ2EgZGUgbG9zIGFyY2hpdm9zIENERiBkZSBsYSBtaXNpb24gQUNFLgoKTUFHICAgID0gbWFnbmV0b21ldHJvIGRlIEFDRSAoY2FtcG8gbWFnbmV0aWNvLCBuVCkKU1dFUEFNID0gU29sYXIgV2luZCBFbGVjdHJvbiBQcm90b24gQWxwaGEgTW9uaXRvciAocGxhc21hKQoKTG9zIENERiBtYXJjYW4gbG9zIGRhdG9zIGludmFsaWRvcyBjb24gdmFsb3JlcyBjZW50aW5lbGEgZW5vcm1lbWVudGUgbmVnYXRpdm9zCihkZWwgb3JkZW4gZGUgLTFlMzEpIGVuIHZleiBkZSBkZWphcmxvcyB2YWNpb3MuIFNpIG5vIHNlIGNvbnZpZXJ0ZW4gYSBOYU4gYW50ZXMKZGUgcHJvbWVkaWFyLCBjb250YW1pbmFuIHRvZG8gZWwgYW5hbGlzaXMgZW4gc2lsZW5jaW8uCgpOT1RBOiBsb3Mgbm9tYnJlcyBkZSB2YXJpYWJsZSBkZSBhYmFqbyBzb24gbG9zIHF1ZSB1c2EgZWwgbm90ZWJvb2sgb3JpZ2luYWwgeSBubwpzZSBoYW4gcG9kaWRvIHZlcmlmaWNhciBjb250cmEgdW4gYXJjaGl2byByZWFsIHRvZGF2aWEgKHZlciBzZWNjaW9uIDcgZGVsCmNlcmVicm86IGxvcyBzZXJ2aWRvcmVzIGRlIGxhIE5BU0EgZXN0YW4gYmxvcXVlYWRvcyB5IGxvcyBkYXRvcyBhdW4gbm8gbGxlZ2FuKS4KU2kgYWwgYWJyaXIgdW4gQ0RGIHJlYWwgYWxndW4gbm9tYnJlIG5vIGNvaW5jaWRlLCBgaW5zcGVjdF9jZGZgIGxvcyBsaXN0YS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgojIFVtYnJhbCBkZSB2YWxvciBjZW50aW5lbGEuIEN1YWxxdWllciBjb3NhIHBvciBkZWJham8gZXMgZGF0byBpbnZhbGlkby4KRklMTF9WQUxVRSA9IC0xZTMwCgpWQVJJQUJMRVNfSU1GID0gWyJCeCIsICJCeSIsICJCeiJdClZBUklBQkxFU19TV0UgPSBbIlZ4IiwgIlZ5IiwgIlZ6IiwgIkRlbnNpZGFkIiwgIlRlbXBlcmF0dXJhIiwgIkFscGhhIHRvIHByb3RvbiBkZW5zaXR5Il0KCgpkZWYgaW5zcGVjdF9jZGYocGF0aDogc3RyIHwgUGF0aCkgLT4gbGlzdFtzdHJdOgogICAgIiIiTGlzdGEgbGFzIHZhcmlhYmxlcyBxdWUgY29udGllbmUgdW4gQ0RGLCBwYXJhIHZlcmlmaWNhciBsb3Mgbm9tYnJlcy4iIiIKICAgIGZyb20gY2RmbGliIGltcG9ydCBDREYKCiAgICByZXR1cm4gbGlzdChDREYoc3RyKHBhdGgpKS5jZGZfaW5mbygpLnpWYXJpYWJsZXMpCgoKZGVmIF9saW1waWFyKGRmOiBwZC5EYXRhRnJhbWUsICosIGNvbHVtbmFfdGllbXBvOiBzdHIgPSAiVGllbXBvIikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiQ29udmllcnRlIGxvcyB2YWxvcmVzIGNlbnRpbmVsYSBhIE5hTi4iIiIKICAgIGNvbHMgPSBkZi5jb2x1bW5zLmRyb3AoY29sdW1uYV90aWVtcG8pCiAgICBkZltjb2xzXSA9IGRmW2NvbHNdLndoZXJlKGRmW2NvbHNdID4gRklMTF9WQUxVRSwgbnAubmFuKQogICAgcmV0dXJuIGRmCgoKZGVmIGxvYWRfaW1mKHBhdGg6IHN0ciB8IFBhdGgpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIkNhbXBvIG1hZ25ldGljbyBpbnRlcnBsYW5ldGFyaW86IFRpZW1wbywgQngsIEJ5LCBCei4iIiIKICAgIGZyb20gY2RmbGliIGltcG9ydCBDREYsIGNkZmVwb2NoCgogICAgY2RmID0gQ0RGKHN0cihwYXRoKSkKICAgIGIgPSBucC5hc2FycmF5KGNkZlsiQkdTRWMiXSkKICAgIGRmID0gcGQuRGF0YUZyYW1lKAogICAgICAgIHsKICAgICAgICAgICAgIlRpZW1wbyI6IGNkZmVwb2NoLnRvX2RhdGV0aW1lKGNkZlsiRXBvY2giXSksCiAgICAgICAgICAgICJCeCI6IGJbOiwgMF0sCiAgICAgICAgICAgICJCeSI6IGJbOiwgMV0sCiAgICAgICAgICAgICJCeiI6IGJbOiwgMl0sCiAgICAgICAgfQogICAgKQogICAgcmV0dXJuIF9saW1waWFyKGRmKQoKCmRlZiBsb2FkX3N3ZShwYXRoOiBzdHIgfCBQYXRoKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJQbGFzbWEgZGVsIHZpZW50byBzb2xhcjogVGllbXBvLCBWeCwgVnksIFZ6LCBkZW5zaWRhZCwgdGVtcGVyYXR1cmEsIGFscGhhLiIiIgogICAgZnJvbSBjZGZsaWIgaW1wb3J0IENERiwgY2RmZXBvY2gKCiAgICBjZGYgPSBDREYoc3RyKHBhdGgpKQogICAgdiA9IG5wLmFzYXJyYXkoY2RmWyJWX0dTRSJdKQogICAgZGYgPSBwZC5EYXRhRnJhbWUoCiAgICAgICAgewogICAgICAgICAgICAiVGllbXBvIjogY2RmZXBvY2gudG9fZGF0ZXRpbWUoY2RmWyJFcG9jaCJdKSwKICAgICAgICAgICAgIlZ4Ijogdls6LCAwXSwKICAgICAgICAgICAgIlZ5Ijogdls6LCAxXSwKICAgICAgICAgICAgIlZ6Ijogdls6LCAyXSwKICAgICAgICAgICAgIkRlbnNpZGFkIjogbnAuYXNhcnJheShjZGZbIk5wIl0pLAogICAgICAgICAgICAiVGVtcGVyYXR1cmEiOiBucC5hc2FycmF5KGNkZlsiVHByIl0pLAogICAgICAgICAgICAiQWxwaGEgdG8gcHJvdG9uIGRlbnNpdHkiOiBucC5hc2FycmF5KGNkZlsiYWxwaGFfcmF0aW8iXSksCiAgICAgICAgfQogICAgKQogICAgcmV0dXJuIF9saW1waWFyKGRmKQoKCmRlZiBsb2FkX21hbnkocGF0aHM6IGxpc3Rbc3RyIHwgUGF0aF0sIGNhcmdhZG9yKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJDYXJnYSB2YXJpb3MgQ0RGIGRlbCBtaXNtbyB0aXBvIHkgbG9zIGNvbmNhdGVuYSBvcmRlbmFkb3MgcG9yIHRpZW1wby4KCiAgICBMb3MgcGVyaW9kb3MgcXVlZGFuIHBlZ2Fkb3MgZW4gdW4gc29sbyBEYXRhRnJhbWUsIHBlcm8gZXNvIE5PIGNyZWEgdmVudGFuYXMKICAgIHF1ZSBjcnVjZW4gZWwgaHVlY286IGBmZWF0dXJlcy53aW5kb3dzLm1ha2Vfd2luZG93c2AgY29ydGEgcG9yIHRyYW1vcwogICAgY29udGludW9zIGFudGVzIGRlIGdlbmVyYXJsYXMuCiAgICAiIiIKICAgIHBhcnRlcyA9IFtjYXJnYWRvcihwKSBmb3IgcCBpbiBwYXRoc10KICAgIGlmIG5vdCBwYXJ0ZXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTm8gc2UgcmVjaWJpbyBuaW5ndW4gYXJjaGl2byBwYXJhIGNhcmdhciIpCiAgICByZXR1cm4gKAogICAgICAgIHBkLmNvbmNhdChwYXJ0ZXMsIGlnbm9yZV9pbmRleD1UcnVlKQogICAgICAgIC5zb3J0X3ZhbHVlcygiVGllbXBvIikKICAgICAgICAuZHJvcF9kdXBsaWNhdGVzKHN1YnNldD0iVGllbXBvIikKICAgICAgICAucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgKQoKCmRlZiByZXNhbXBsZV80bWluKGRmOiBwZC5EYXRhRnJhbWUsICosIGNvbHVtbmFfdGllbXBvOiBzdHIgPSAiVGllbXBvIikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiUHJvbWVkaWEgYSBpbnRlcnZhbG9zIGRlIDQgbWludXRvcy4KCiAgICBTV0UgdmllbmUgYSB+NjQgc2VndW5kb3MgZSBJTUYgYSAxIG1pbnV0bzogaGF5IHF1ZSBsbGV2YXJsb3MgYSB1bmEgY2FkZW5jaWEKICAgIGNvbXVuIGFudGVzIGRlIGNydXphcmxvcy4gYC5tZWFuKClgIGlnbm9yYSBsb3MgTmFOIHBvciBkZWZlY3RvLgogICAgIiIiCiAgICByZXR1cm4gKAogICAgICAgIGRmLnNldF9pbmRleChjb2x1bW5hX3RpZW1wbykKICAgICAgICAucmVzYW1wbGUoIjRtaW4iKQogICAgICAgIC5tZWFuKCkKICAgICAgICAucmVzZXRfaW5kZXgoKQogICAgKQoKCmRlZiBtZXJnZV9pbWZfc3dlKAogICAgaW1mOiBwZC5EYXRhRnJhbWUsIHN3ZV80bWluOiBwZC5EYXRhRnJhbWUsICosIGNvbHVtbmFfdGllbXBvOiBzdHIgPSAiVGllbXBvIgopIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIkNydXphIGNhbXBvIG1hZ25ldGljbyB5IHBsYXNtYSBwb3IgdGllbXBvIGV4YWN0byAoaW5uZXIgam9pbikuIiIiCiAgICByZXR1cm4gcGQubWVyZ2UoaW1mLCBzd2VfNG1pbiwgb249Y29sdW1uYV90aWVtcG8sIGhvdz0iaW5uZXIiKS5yZXNldF9pbmRleChkcm9wPVRydWUpCgoKZGVmIGFkZF92ZWxvY2l0eV9tYWduaXR1ZGUoZGY6IHBkLkRhdGFGcmFtZSkgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiQWdyZWdhIHxWfCB5IHN1IHZhcmlhY2lvbiBwdW50byBhIHB1bnRvLiIiIgogICAgb3V0ID0gZGYuY29weSgpCiAgICBvdXRbIlZfbWFnIl0gPSBucC5zcXJ0KG91dFsiVngiXSAqKiAyICsgb3V0WyJWeSJdICoqIDIgKyBvdXRbIlZ6Il0gKiogMikKICAgIG91dFsiVl9tYWdfdHJlbmQiXSA9IG91dFsiVl9tYWciXS5kaWZmKCkKICAgIHJldHVybiBvdXQK",
    "tesis/data/catalogs.py": "IiIiQ2FyZ2EgZGUgbG9zIGNhdGFsb2dvcyBkZSBldmVudG9zIHF1ZSBzaXJ2ZW4gZGUgdmVyZGFkIGRlIHJlZmVyZW5jaWEuCgpJQ01FOiBjYXRhbG9nbyB0aXBvIFJpY2hhcmRzb24gJiBDYW5lLCBjb24gY29sdW1uYXMgJ1N0YXJ0JyB5ICdFbmQnIGVuIGZvcm1hdG8KICAgICAgJyVZLyVtLyVkICVIJU0nLgpTSVI6ICBjYXRhbG9nbyBjb24gY29sdW1uYXMgJ3N0YXJ0X3RpbWUnIHkgJ2VuZF90aW1lJyBlbiBmb3JtYXRvIHZhcmlhYmxlLgoKQW1ib3Mgc2Ugbm9ybWFsaXphbiBhIHVuIERhdGFGcmFtZSBjb24gZXhhY3RhbWVudGUgZG9zIGNvbHVtbmFzIGRhdGV0aW1lLAonc3RhcnRfdGltZScgeSAnZW5kX3RpbWUnLCBxdWUgZXMgbG8gcXVlIGVzcGVyYSBtb2RlbHMubGFiZWxpbmcuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgcGFuZGFzIGFzIHBkCgpDT0xVTU5BUyA9IFsic3RhcnRfdGltZSIsICJlbmRfdGltZSJdCgoKZGVmIF92YWxpZGFyKGRmOiBwZC5EYXRhRnJhbWUsIG9yaWdlbjogc3RyKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJEZXNjYXJ0YSBmaWxhcyBzaW4gZmVjaGEgdmFsaWRhIG8gY29uIGVsIGludGVydmFsbyBpbnZlcnRpZG8uIiIiCiAgICBhbnRlcyA9IGxlbihkZikKICAgIGRmID0gZGYuZHJvcG5hKHN1YnNldD1DT0xVTU5BUykKICAgIGRmID0gZGZbZGZbImVuZF90aW1lIl0gPiBkZlsic3RhcnRfdGltZSJdXQogICAgZGVzY2FydGFkYXMgPSBhbnRlcyAtIGxlbihkZikKICAgIGlmIGRlc2NhcnRhZGFzOgogICAgICAgIHByaW50KGYie29yaWdlbn06IHtkZXNjYXJ0YWRhc30gZXZlbnRvKHMpIGRlc2NhcnRhZG8ocykgcG9yIGZlY2hhcyBpbnZhbGlkYXMiKQogICAgcmV0dXJuIGRmLnNvcnRfdmFsdWVzKCJzdGFydF90aW1lIikucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQoKCmRlZiBsb2FkX2ljbWUocGF0aDogc3RyIHwgUGF0aCkgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiQ2F0YWxvZ28gSUNNRSAtPiBEYXRhRnJhbWUgY29uIHN0YXJ0X3RpbWUgLyBlbmRfdGltZS4iIiIKICAgIHJhdyA9IHBkLnJlYWRfY3N2KHBhdGgpCiAgICBkZiA9IHBkLkRhdGFGcmFtZSgKICAgICAgICB7CiAgICAgICAgICAgICJzdGFydF90aW1lIjogcGQudG9fZGF0ZXRpbWUoCiAgICAgICAgICAgICAgICByYXdbIlN0YXJ0Il0sIGZvcm1hdD0iJVkvJW0vJWQgJUglTSIsIGVycm9ycz0iY29lcmNlIgogICAgICAgICAgICApLAogICAgICAgICAgICAiZW5kX3RpbWUiOiBwZC50b19kYXRldGltZShyYXdbIkVuZCJdLCBmb3JtYXQ9IiVZLyVtLyVkICVIJU0iLCBlcnJvcnM9ImNvZXJjZSIpLAogICAgICAgIH0KICAgICkKICAgIHJldHVybiBfdmFsaWRhcihkZiwgIklDTUUiKQoKCmRlZiBsb2FkX3NpcihwYXRoOiBzdHIgfCBQYXRoKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJDYXRhbG9nbyBTSVIgLT4gRGF0YUZyYW1lIGNvbiBzdGFydF90aW1lIC8gZW5kX3RpbWUuIiIiCiAgICByYXcgPSBwZC5yZWFkX2NzdihwYXRoKQogICAgZGYgPSBwZC5EYXRhRnJhbWUoCiAgICAgICAgewogICAgICAgICAgICAic3RhcnRfdGltZSI6IHBkLnRvX2RhdGV0aW1lKAogICAgICAgICAgICAgICAgcmF3WyJzdGFydF90aW1lIl0sIGZvcm1hdD0ibWl4ZWQiLCBlcnJvcnM9ImNvZXJjZSIKICAgICAgICAgICAgKSwKICAgICAgICAgICAgImVuZF90aW1lIjogcGQudG9fZGF0ZXRpbWUocmF3WyJlbmRfdGltZSJdLCBmb3JtYXQ9Im1peGVkIiwgZXJyb3JzPSJjb2VyY2UiKSwKICAgICAgICB9CiAgICApCiAgICByZXR1cm4gX3ZhbGlkYXIoZGYsICJTSVIiKQoKCmRlZiBmaWx0cmFyX3BlcmlvZG8oCiAgICBjYXRhbG9nbzogcGQuRGF0YUZyYW1lLCBpbmljaW86IHN0ciB8IHBkLlRpbWVzdGFtcCwgZmluOiBzdHIgfCBwZC5UaW1lc3RhbXAKKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJSZWNvcnRhIGVsIGNhdGFsb2dvIGEgbG9zIGV2ZW50b3MgcXVlIHRvY2FuIGVsIHJhbmdvIHBlZGlkby4KCiAgICBTZSBjb25zZXJ2YSBjdWFscXVpZXIgZXZlbnRvIHF1ZSBzb2xhcGUgZWwgcmFuZ28sIG5vIHNvbG8gbG9zIHF1ZSBlbXBpZXphbgogICAgZGVudHJvOiB1biBJQ01FIHF1ZSBhcnJhbmNhIGVsIDMxIGRlIGRpY2llbWJyZSBzaWd1ZSBzaWVuZG8gcmVsZXZhbnRlIHBhcmEKICAgIGxhcyB2ZW50YW5hcyBkZWwgMSBkZSBlbmVyby4KICAgICIiIgogICAgaW5pY2lvLCBmaW4gPSBwZC5UaW1lc3RhbXAoaW5pY2lvKSwgcGQuVGltZXN0YW1wKGZpbikKICAgIHJldHVybiBjYXRhbG9nb1sKICAgICAgICAoY2F0YWxvZ29bImVuZF90aW1lIl0gPj0gaW5pY2lvKSAmIChjYXRhbG9nb1sic3RhcnRfdGltZSJdIDw9IGZpbikKICAgIF0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQo=",
    "tesis/features/windows.py": "IiIiVmVudGFuYXMgZGVzbGl6YW50ZXMgc29icmUgbGFzIHNlcmllcyBkZWwgdmllbnRvIHNvbGFyLgoKQXF1aSB2aXZlbiBkb3MgY29ycmVjY2lvbmVzIHJlc3BlY3RvIGFsIG5vdGVib29rIG9yaWdpbmFsOgoKQlVHIDQgLSBWZW50YW5hcyBxdWUgY3J1emFiYW4gZWwgaHVlY28gMjAwOCAtPiAyMDE1LgogICAgRWwgb3JpZ2luYWwgaGFjaWEgY29uY2F0KFsyMDA4LCAyMDE1XSkgeSBjb3JyaWEgc2xpZGluZ193aW5kb3dfdmlldyBzb2JyZQogICAgdG9kbyBlbCBhcnJheS4gRXNvIGdlbmVyYSB2ZW50YW5hcyBkZSAyMCBwdW50b3MgY29udGlndW9zIGVuIG1lbW9yaWEgcGVybwogICAgc2VwYXJhZG9zIHBvciA3IGFuaW9zIGVuIGVsIHRpZW1wbywgcXVlIHNvbiBmaXNpY2FtZW50ZSBiYXN1cmEgeSBwcm9kdWNpYW4KICAgIGxhIGV0aXF1ZXRhICdVbmtub3duX1llYXInLiBBcXVpIGxhcyB2ZW50YW5hcyBzZSBnZW5lcmFuIHBvciBzZWdtZW50bwogICAgdGVtcG9yYWwgY29udGludW8gKHZlciBgc3BsaXRfc2VnbWVudHNgKSwgYXNpIHF1ZSBudW5jYSBjcnV6YW4gdW4gaHVlY28uCgpSRU5ESU1JRU5UTyAtIExhIGludGVycG9sYWNpb24gdGVuaWEgdW4gZG9ibGUgYnVjbGUgUHl0aG9uIChucyB4IG5mLCBjaWVudG9zCiAgICBkZSBtaWxlcyBkZSBpdGVyYWNpb25lcykuIGBpbnRlcnBvbGF0ZV93aW5kb3dzYCBsYSBoYWNlIHZlY3Rvcml6YWRhLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gbnVtcHkubGliLnN0cmlkZV90cmlja3MgaW1wb3J0IHNsaWRpbmdfd2luZG93X3ZpZXcKCiMgQ2FkZW5jaWEgbm9taW5hbCB0cmFzIGVsIHJlc2FtcGxlOiB1biBwdW50byBjYWRhIDQgbWludXRvcy4KQ0FERU5DSUEgPSBwZC5UaW1lZGVsdGEoIjRtaW4iKQoKCmRlZiBzcGxpdF9zZWdtZW50cygKICAgIHRpbWVzOiBwZC5TZXJpZXMsICosIGNhZGVuY2lhOiBwZC5UaW1lZGVsdGEgPSBDQURFTkNJQSwgdG9sZXJhbmNpYTogZmxvYXQgPSAxLjUKKSAtPiBsaXN0W3R1cGxlW2ludCwgaW50XV06CiAgICAiIiJDb3J0YSBsYSBzZXJpZSBlbiB0cmFtb3MgdGVtcG9yYWxtZW50ZSBjb250aW51b3MuCgogICAgRGV2dWVsdmUgdW5hIGxpc3RhIGRlIHBhcmVzIChpbmljaW8sIGZpbikgZGUgaW5kaWNlcyBwb3NpY2lvbmFsZXMsIGNvbiBgZmluYAogICAgZXhjbHVzaXZvLiBVbiBodWVjbyBtYXlvciBxdWUgYGNhZGVuY2lhICogdG9sZXJhbmNpYWAgYWJyZSB1biB0cmFtbyBudWV2by4KCiAgICBFc3RvIGVzIGxvIHF1ZSBpbXBpZGUgcXVlIHVuYSB2ZW50YW5hIGVtcGllY2UgZW4gZGljaWVtYnJlIDIwMDggeSB0ZXJtaW5lIGVuCiAgICBlbmVybyAyMDE1LgogICAgIiIiCiAgICBpZiBsZW4odGltZXMpID09IDA6CiAgICAgICAgcmV0dXJuIFtdCgogICAgdCA9IHBkLnRvX2RhdGV0aW1lKHBkLlNlcmllcyh0aW1lcykucmVzZXRfaW5kZXgoZHJvcD1UcnVlKSkKICAgIGRlbHRhcyA9IHQuZGlmZigpCiAgICAjIEVsIHByaW1lciBkZWx0YSBlcyBOYVQ7IG5vIG1hcmNhIGNvcnRlLgogICAgY29ydGVzID0gbnAuZmxhdG5vbnplcm8oKGRlbHRhcyA+IGNhZGVuY2lhICogdG9sZXJhbmNpYSkudG9fbnVtcHkoKVsxOl0pICsgMQoKICAgIGJvcmRlcyA9IFswLCAqKGludChjKSBmb3IgYyBpbiBjb3J0ZXMpLCBsZW4odCldCiAgICByZXR1cm4gWyhib3JkZXNbaV0sIGJvcmRlc1tpICsgMV0pIGZvciBpIGluIHJhbmdlKGxlbihib3JkZXMpIC0gMSldCgoKZGVmIG1ha2Vfd2luZG93cygKICAgIGRmOiBwZC5EYXRhRnJhbWUsCiAgICB2YXJpYWJsZXM6IGxpc3Rbc3RyXSwKICAgICosCiAgICB3aW5kb3dfc2l6ZTogaW50ID0gMjAsCiAgICBzdHJpZGU6IGludCA9IDEsCiAgICBjb2x1bW5hX3RpZW1wbzogc3RyID0gIlRpZW1wbyIsCikgLT4gdHVwbGVbbnAubmRhcnJheSwgbnAubmRhcnJheV06CiAgICAiIiJWZW50YW5hcyBkZXNsaXphbnRlcyBxdWUgTk8gY3J1emFuIGh1ZWNvcyB0ZW1wb3JhbGVzLgoKICAgIERldnVlbHZlOgogICAgICAgIHdpbmRvd3M6IChuX3ZlbnRhbmFzLCB3aW5kb3dfc2l6ZSwgbl92YXJpYWJsZXMpCiAgICAgICAgc3RhcnRzOiAgKG5fdmVudGFuYXMsKSBpbmRpY2UgcG9zaWNpb25hbCBlbiBgZGZgIGRvbmRlIGFycmFuY2EgY2FkYSB2ZW50YW5hLgoKICAgIGBzdGFydHNgIGVzIGxvIHF1ZSBkZXNwdWVzIHBlcm1pdGUgcmVjdXBlcmFyIGxvcyB0aWVtcG9zIGRlIGluaWNpbyB5IGZpbi4KICAgIFNlIGRldnVlbHZlIGV4cGxpY2l0YW1lbnRlIGVuIHZleiBkZSByZWNvbnN0cnVpcmxvIGFzdW1pZW5kbyBzdHJpZGU9MSwgcXVlCiAgICBlcmEgb3RyYSBmcmFnaWxpZGFkIGRlbCBvcmlnaW5hbC4KICAgICIiIgogICAgZmFsdGFudGVzID0gW3YgZm9yIHYgaW4gdmFyaWFibGVzIGlmIHYgbm90IGluIGRmLmNvbHVtbnNdCiAgICBpZiBmYWx0YW50ZXM6CiAgICAgICAgcmFpc2UgS2V5RXJyb3IoZiJGYWx0YW4gY29sdW1uYXMgZW4gZWwgRGF0YUZyYW1lOiB7ZmFsdGFudGVzfSIpCgogICAgZGF0YSA9IGRmW3ZhcmlhYmxlc10udG9fbnVtcHkoZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGJsb3F1ZXM6IGxpc3RbbnAubmRhcnJheV0gPSBbXQogICAgc3RhcnRzOiBsaXN0W25wLm5kYXJyYXldID0gW10KCiAgICBmb3IgaW5pLCBmaW4gaW4gc3BsaXRfc2VnbWVudHMoZGZbY29sdW1uYV90aWVtcG9dKToKICAgICAgICB0cmFtbyA9IGRhdGFbaW5pOmZpbl0KICAgICAgICBpZiBsZW4odHJhbW8pIDwgd2luZG93X3NpemU6CiAgICAgICAgICAgIGNvbnRpbnVlICAjIHRyYW1vIGRlbWFzaWFkbyBjb3J0byBwYXJhIHVuYSBzb2xhIHZlbnRhbmEKICAgICAgICB2aXN0YXMgPSBzbGlkaW5nX3dpbmRvd192aWV3KHRyYW1vLCB3aW5kb3dfc2hhcGU9d2luZG93X3NpemUsIGF4aXM9MCkKICAgICAgICAjIHNsaWRpbmdfd2luZG93X3ZpZXcgc29icmUgYXhpcz0wIGRldnVlbHZlIChuLCBuZiwgd3NpemUpOiByZW9yZGVuYW1vcy4KICAgICAgICB2aXN0YXMgPSB2aXN0YXMudHJhbnNwb3NlKDAsIDIsIDEpWzo6c3RyaWRlXQogICAgICAgIGJsb3F1ZXMuYXBwZW5kKG5wLmFzY29udGlndW91c2FycmF5KHZpc3RhcykpCiAgICAgICAgc3RhcnRzLmFwcGVuZChpbmkgKyBucC5hcmFuZ2UoMCwgbGVuKHRyYW1vKSAtIHdpbmRvd19zaXplICsgMSwgc3RyaWRlKSkKCiAgICBpZiBub3QgYmxvcXVlczoKICAgICAgICB2YWNpbyA9IG5wLmVtcHR5KCgwLCB3aW5kb3dfc2l6ZSwgbGVuKHZhcmlhYmxlcykpLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgICAgIHJldHVybiB2YWNpbywgbnAuZW1wdHkoMCwgZHR5cGU9bnAuaW50NjQpCgogICAgcmV0dXJuIG5wLmNvbmNhdGVuYXRlKGJsb3F1ZXMpLCBucC5jb25jYXRlbmF0ZShzdGFydHMpCgoKZGVmIG1heF9jb25zZWN1dGl2ZV9uYW5zKHdpbmRvd3M6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJSYWNoYSBtYXhpbWEgZGUgTmFOIHBvciB2ZW50YW5hIHkgdmFyaWFibGUuCgogICAgd2luZG93czogKG5zLCB3c2l6ZSwgbmYpICAtPiAgZGV2dWVsdmUgKG5zLCBuZikKICAgICIiIgogICAgbmFuID0gbnAuaXNuYW4od2luZG93cykKICAgIF8sIHdzaXplLCBfID0gbmFuLnNoYXBlCiAgICBpZHggPSBucC5hcmFuZ2Uod3NpemUpW05vbmUsIDosIE5vbmVdCiAgICB1bHRpbW9fdmFsaWRvID0gbnAubWF4aW11bS5hY2N1bXVsYXRlKG5wLndoZXJlKG5hbiwgLTEsIGlkeCksIGF4aXM9MSkKICAgIHJldHVybiBucC53aGVyZShuYW4sIGlkeCAtIHVsdGltb192YWxpZG8sIDApLm1heChheGlzPTEpCgoKZGVmIGRyb3BfZ2FwcHlfd2luZG93cygKICAgIHdpbmRvd3M6IG5wLm5kYXJyYXksIHN0YXJ0czogbnAubmRhcnJheSwgKiwgbWF4X2NvbnNlYzogaW50ID0gMQopIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgIiIiRGVzY2FydGEgbGFzIHZlbnRhbmFzIGNvbiByYWNoYXMgZGUgTmFOIGRlbWFzaWFkbyBsYXJnYXMuCgogICAgYG1heF9jb25zZWM9MWAgY29uc2VydmEgbGFzIHZlbnRhbmFzIGNvbiBOYU4gc3VlbHRvcyAoaW50ZXJwb2xhYmxlcykgeQogICAgZGVzY2FydGEgbGFzIHF1ZSB0aWVuZW4gMiBvIG1hcyBzZWd1aWRvcywgaWd1YWwgcXVlIGVsIG5vdGVib29rIG9yaWdpbmFsLgoKICAgIEVzdGUgY3JpdGVyaW8gdGllbmUgc2VudGlkbyBjdWFuZG8gY2FkYSBwdW50byBkZSBsYSB2ZW50YW5hIHZhIGEgdXNhcnNlOgogICAgZG9zIGh1ZWNvcyBzZWd1aWRvcyBvYmxpZ2FuIGEgaW52ZW50YXIgZGVtYXNpYWRvIGFsIGludGVycG9sYXIuIFNpIGVuCiAgICBjYW1iaW8gbGEgdmVudGFuYSBzZSB2YSBhIHJlc3VtaXIgZW4gZXN0YWRpc3RpY29zIGludGVncmFsZXMgKG1lZGlhbmEsCiAgICBkZXN2aWFjaW9uLCB0ZW5kZW5jaWEpLCBsbyBxdWUgaW1wb3J0YSBubyBlcyBxdWUgbG9zIGh1ZWNvcyBzZWFuCiAgICBjb25zZWN1dGl2b3Mgc2lubyBjdWFudG9zIGRhdG9zIGJ1ZW5vcyBxdWVkYW46IHZlciBgZHJvcF9zcGFyc2Vfd2luZG93c2AuCiAgICAiIiIKICAgIHBlb3IgPSBtYXhfY29uc2VjdXRpdmVfbmFucyh3aW5kb3dzKS5tYXgoYXhpcz0xKQogICAgY29uc2VydmFyID0gcGVvciA8PSBtYXhfY29uc2VjCiAgICByZXR1cm4gd2luZG93c1tjb25zZXJ2YXJdLCBzdGFydHNbY29uc2VydmFyXQoKCmRlZiBmcmFjY2lvbl92YWxpZGEod2luZG93czogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgICIiIkZyYWNjaW9uIGRlIHB1bnRvcyBubyBhdXNlbnRlcyBwb3IgdmVudGFuYSB5IHZhcmlhYmxlLgoKICAgIHdpbmRvd3M6IChucywgd3NpemUsIG5mKSAgLT4gIGRldnVlbHZlIChucywgbmYpIGVuIFswLCAxXQogICAgIiIiCiAgICByZXR1cm4gMS4wIC0gbnAuaXNuYW4od2luZG93cykubWVhbihheGlzPTEpCgoKZGVmIGRyb3Bfc3BhcnNlX3dpbmRvd3MoCiAgICB3aW5kb3dzOiBucC5uZGFycmF5LCBzdGFydHM6IG5wLm5kYXJyYXksICosIG1pbmltbzogZmxvYXQgPSAwLjc1CikgLT4gdHVwbGVbbnAubmRhcnJheSwgbnAubmRhcnJheV06CiAgICAiIiJEZXNjYXJ0YSBsYXMgdmVudGFuYXMgc2luIGRhdG9zIHN1ZmljaWVudGVzIHBhcmEgdW4gZXN0YWRpc3RpY28gZmlhYmxlLgoKICAgIEFsdGVybmF0aXZhIGEgYGRyb3BfZ2FwcHlfd2luZG93c2AgcGVuc2FkYSBwYXJhIGRlc2NyaXB0b3JlcyBpbnRlZ3JhbGVzOiBubwogICAgaW1wb3J0YSBxdWUgbG9zIGh1ZWNvcyBlc3RlbiBzZWd1aWRvcywgc2lubyBxdWUgcXVlZGUgYWwgbWVub3MgdW5hIGZyYWNjaW9uCiAgICBgbWluaW1vYCBkZSBwdW50b3MgdmFsaWRvcyBlbiBDQURBIHZhcmlhYmxlLiBVbmEgbWVkaWFuYSBzb2JyZSAxNSBkZSAyMAogICAgcHVudG9zIGVzIHBlcmZlY3RhbWVudGUgdXRpbGl6YWJsZTsgZWwgY3JpdGVyaW8gZGUgcmFjaGFzIGxhIGRlc2NhcnRhcmlhIHNpCiAgICBkb3MgZGUgZXNvcyBodWVjb3MgZnVlc2VuIGNvbnNlY3V0aXZvcy4KICAgICIiIgogICAgaWYgbm90IDAgPCBtaW5pbW8gPD0gMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYibWluaW1vIGRlYmUgZXN0YXIgZW4gKDAsIDFdLCBsbGVnbyB7bWluaW1vfSIpCiAgICBjb25zZXJ2YXIgPSBmcmFjY2lvbl92YWxpZGEod2luZG93cykubWluKGF4aXM9MSkgPj0gbWluaW1vCiAgICByZXR1cm4gd2luZG93c1tjb25zZXJ2YXJdLCBzdGFydHNbY29uc2VydmFyXQoKCiMgTWVtb3JpYSBtYXhpbWEgcXVlIHB1ZWRlIG9jdXBhciB1biBibG9xdWUgZHVyYW50ZSBsYSBpbnRlcnBvbGFjaW9uLgojIEVsIGFsZ29yaXRtbyBjcmVhIGRlbCBvcmRlbiBkZSAxNiBhcnJheXMgdGVtcG9yYWxlcyBkZWwgdGFtYW5vIGRlbCBibG9xdWUsCiMgYXNpIHF1ZSBlbCBwaWNvIHJlYWwgcm9uZGEgR0JfUE9SX0JMT1FVRSAqIDE2LgpHQl9QT1JfQkxPUVVFID0gMC4wNgoKCmRlZiBfaW50ZXJwb2xhcl9ibG9xdWUoYmxvcXVlOiBucC5uZGFycmF5KSAtPiBucC5uZGFycmF5OgogICAgIiIiSW50ZXJwb2xhY2lvbiBsaW5lYWwgc29icmUgZWwgZWplIHRlbXBvcmFsIGRlIHVuIGJsb3F1ZSBkZSB2ZW50YW5hcy4iIiIKICAgIG91dCA9IGJsb3F1ZS5hc3R5cGUobnAuZmxvYXQ2NCwgY29weT1UcnVlKQogICAgbmFuID0gbnAuaXNuYW4ob3V0KQogICAgaWYgbm90IG5hbi5hbnkoKToKICAgICAgICByZXR1cm4gb3V0CgogICAgXywgd3NpemUsIF8gPSBvdXQuc2hhcGUKICAgIGlkeCA9IG5wLmFyYW5nZSh3c2l6ZSwgZHR5cGU9bnAuaW50MzIpW05vbmUsIDosIE5vbmVdCgogICAgIyBJbmRpY2UgZGVsIHVsdGltbyB2YWxpZG8gYSBsYSBpenF1aWVyZGEgKC0xIHNpIG5vIGhheSBuaW5ndW5vKS4KICAgIGl6cSA9IG5wLm1heGltdW0uYWNjdW11bGF0ZShucC53aGVyZShuYW4sIG5wLmludDMyKC0xKSwgaWR4KSwgYXhpcz0xKQogICAgIyBJbmRpY2UgZGVsIHByaW1lciB2YWxpZG8gYSBsYSBkZXJlY2hhICh3c2l6ZSBzaSBubyBoYXkgbmluZ3VubykuCiAgICBkZXIgPSBucC5taW5pbXVtLmFjY3VtdWxhdGUoCiAgICAgICAgbnAud2hlcmUobmFuLCBucC5pbnQzMih3c2l6ZSksIGlkeClbOiwgOjotMSwgOl0sIGF4aXM9MQogICAgKVs6LCA6Oi0xLCA6XQoKICAgIGhheV9penEgPSBpenEgPj0gMAogICAgaGF5X2RlciA9IGRlciA8IHdzaXplCgogICAgbnAuY2xpcChpenEsIDAsIHdzaXplIC0gMSwgb3V0PWl6cSkKICAgIG5wLmNsaXAoZGVyLCAwLCB3c2l6ZSAtIDEsIG91dD1kZXIpCiAgICB2X2l6cSA9IG5wLnRha2VfYWxvbmdfYXhpcyhvdXQsIGl6cSwgYXhpcz0xKQogICAgdl9kZXIgPSBucC50YWtlX2Fsb25nX2F4aXMob3V0LCBkZXIsIGF4aXM9MSkKCiAgICBzcGFuID0gKGRlciAtIGl6cSkuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICBucC5wdXRtYXNrKHNwYW4sIHNwYW4gPT0gMCwgMS4wKSAgIyBldml0YSAwLzA7IGVsIHBlc28gcXVlZGEgaXJyZWxldmFudGUKICAgIHBlc28gPSAoaWR4IC0gaXpxKSAvIHNwYW4KCiAgICAjIGB2X2RlcmAgeSBgdl9penFgIHNlIHNpZ3VlbiBuZWNlc2l0YW5kbyBhYmFqbyBwYXJhIGxvcyBib3JkZXMsIGFzaSBxdWUgbGEKICAgICMgcmVjdGEgdmEgYSB1biBhcnJheSBwcm9waW8gKGluLXBsYWNlIHNvYnJlIHZfZGVyIGNvcnJvbXBpYSBlbCByZWxsZW5vKS4KICAgIHJlY3RhID0gdl9kZXIgLSB2X2l6cQogICAgcmVjdGEgKj0gcGVzbwogICAgcmVjdGEgKz0gdl9penEKCiAgICByZWxsZW5vID0gbnAud2hlcmUoaGF5X2l6cSAmIGhheV9kZXIsIHJlY3RhLCBucC53aGVyZShoYXlfaXpxLCB2X2l6cSwgdl9kZXIpKQogICAgcmV0dXJuIG5wLndoZXJlKG5hbiwgcmVsbGVubywgb3V0KQoKCmRlZiBpbnRlcnBvbGF0ZV93aW5kb3dzKAogICAgd2luZG93czogbnAubmRhcnJheSwgKiwgZ2JfcG9yX2Jsb3F1ZTogZmxvYXQgPSBHQl9QT1JfQkxPUVVFCikgLT4gbnAubmRhcnJheToKICAgICIiIkludGVycG9sYWNpb24gbGluZWFsIGEgbG8gbGFyZ28gZGVsIGVqZSB0ZW1wb3JhbCwgdmVjdG9yaXphZGEgeSBwb3IgYmxvcXVlcy4KCiAgICBMb3MgTmFOIGRlIGxvcyBleHRyZW1vcyBzZSByZWxsZW5hbiBjb24gZWwgdmFsb3IgdmFsaWRvIG1hcyBjZXJjYW5vCiAgICAoZXF1aXZhbGVudGUgYSBmZmlsbC9iZmlsbCkuIFVuYSB2YXJpYWJsZSBlbnRlcmFtZW50ZSBOYU4gZGVudHJvIGRlIHVuYQogICAgdmVudGFuYSBzZSBxdWVkYSBlbiBOYU46IG5vIGhheSBuYWRhIGNvbiBxdWUgaW50ZXJwb2xhci4KCiAgICBTdXN0aXR1eWUgYWwgZG9ibGUgYnVjbGUgUHl0aG9uIGRlbCBvcmlnaW5hbCwgcXVlIHJlY29ycmlhIGNhZGEgdmVudGFuYSB5CiAgICBjYWRhIHZhcmlhYmxlIHBvciBzZXBhcmFkby4KCiAgICBNRU1PUklBOiBsYSB2ZXJzaW9uIHZlY3Rvcml6YWRhIGNyZWEgZGVsIG9yZGVuIGRlIDE2IGFycmF5cyB0ZW1wb3JhbGVzIGRlbAogICAgdGFtYW5vIGRlIGxhIGVudHJhZGEuIENvbiB2ZW50YW5hcyBkZSAyNCBoIGVzbyBzaWduaWZpY2FiYSB1biBwaWNvIGRlIH4xNyBHQgogICAgcGFyYSB1biB0ZW5zb3IgZGUgMSwzIEdCLCB5IHJldmVudGFiYSBlbiBDb2xhYiB5IGVuIGN1YWxxdWllciBwb3J0YXRpbC4gQXF1aQogICAgc2UgcHJvY2VzYSBwb3IgYmxvcXVlcywgZGUgbW9kbyBxdWUgZWwgcGljbyBkZXBlbmRlIGRlIGBnYl9wb3JfYmxvcXVlYCB5IG5vCiAgICBkZWwgdGFtYW5vIHRvdGFsLiBFbCByZXN1bHRhZG8gZXMgaWRlbnRpY28uCiAgICAiIiIKICAgIGlmIHdpbmRvd3Muc2l6ZSA9PSAwOgogICAgICAgIHJldHVybiB3aW5kb3dzLmFzdHlwZShucC5mbG9hdDY0LCBjb3B5PVRydWUpCgogICAgbnMgPSB3aW5kb3dzLnNoYXBlWzBdCiAgICBieXRlc19wb3JfdmVudGFuYSA9IHdpbmRvd3NbMF0uc2l6ZSAqIDgKICAgIHBvcl9ibG9xdWUgPSBtYXgoMSwgaW50KGdiX3Bvcl9ibG9xdWUgKiAxMDI0KiozIC8gYnl0ZXNfcG9yX3ZlbnRhbmEpKQoKICAgIGlmIHBvcl9ibG9xdWUgPj0gbnM6CiAgICAgICAgcmV0dXJuIF9pbnRlcnBvbGFyX2Jsb3F1ZSh3aW5kb3dzKQoKICAgIG91dCA9IG5wLmVtcHR5KHdpbmRvd3Muc2hhcGUsIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBmb3IgaSBpbiByYW5nZSgwLCBucywgcG9yX2Jsb3F1ZSk6CiAgICAgICAgb3V0W2kgOiBpICsgcG9yX2Jsb3F1ZV0gPSBfaW50ZXJwb2xhcl9ibG9xdWUod2luZG93c1tpIDogaSArIHBvcl9ibG9xdWVdKQogICAgcmV0dXJuIG91dAoKCmRlZiBuYW5fcmVwb3J0KGRmOiBwZC5EYXRhRnJhbWUsIHZhcmlhYmxlczogbGlzdFtzdHJdKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJQb3JjZW50YWplIGRlIE5hTiBwb3IgdmFyaWFibGUsIHBhcmEgZGVjaWRpciBjdWFsZXMgZW50cmFuIGFsIGFuYWxpc2lzLgoKICAgIEJVRyAxOiBlbCBub3RlYm9vayBjYWxjdWxhYmEgZXN0bywgZGVjaWRpYSBlbGltaW5hciAnRGVuc2lkYWQnIHkKICAgICdBbHBoYSB0byBwcm90b24gZGVuc2l0eScuLi4geSBkZXNwdWVzIGxhcyB2b2x2aWEgYSBpbmNsdWlyIGVuIGB2YXJpYWJsZXNgLgogICAgRXN0YSBmdW5jaW9uIGV4aXN0ZSBwYXJhIHF1ZSBsYSBkZWNpc2lvbiBzZWEgZXhwbGljaXRhIHkgcXVlZGUgcmVnaXN0cmFkYSwKICAgIG5vIGltcGxpY2l0YSBlbiB1bmEgbGluZWEgZGUgY29kaWdvIG11ZXJ0by4KICAgICIiIgogICAgc3ViID0gZGZbdmFyaWFibGVzXQogICAgcmV0dXJuICgKICAgICAgICBwZC5EYXRhRnJhbWUoCiAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJuX25hbiI6IHN1Yi5pc25hKCkuc3VtKCksCiAgICAgICAgICAgICAgICAicGN0X25hbiI6IChzdWIuaXNuYSgpLm1lYW4oKSAqIDEwMCkucm91bmQoMiksCiAgICAgICAgICAgIH0KICAgICAgICApCiAgICAgICAgLnNvcnRfdmFsdWVzKCJwY3RfbmFuIiwgYXNjZW5kaW5nPUZhbHNlKQogICAgICAgIC5yZW5hbWVfYXhpcygidmFyaWFibGUiKQogICAgKQo=",
    "tesis/features/summary.py": "IiIiUmVzdW1lbiBkZSBjYWRhIHZlbnRhbmEgYSAobWVkaWFuYSwgZGVzdmlhY2lvbiwgdGVuZGVuY2lhKSBwb3IgdmFyaWFibGUuCgpCVUcgMyAtIEVsIG5vdGVib29rIG9yaWdpbmFsIGVzdGFuZGFyaXphYmEgVFJFUyB2ZWNlczoKICAgIDEuIFN0YW5kYXJkU2NhbGVyIGdsb2JhbCBzb2JyZSBsYXMgdmVudGFuYXMgYXBsYW5hZGFzIChjZWxkYSA0MSkKICAgIDIuIHN0YW5kYXJkaXplX2J5X3llYXIoKSBzb2JyZSBsYXMgZmVhdHVyZXMgcmVzdW1pZGFzIChjZWxkYSA1MSkKICAgIDMuIFN0YW5kYXJkU2NhbGVyIG90cmEgdmV6IGFudGVzIGRlbCBLTWVhbnMgKGNlbGRhIDYwKQoKICAgIEFkZW1hcyBlbCBwcmltZXJvIGFqdXN0YWJhIHNvYnJlIDIwMDggeSAyMDE1IGp1bnRvcywgcXVlIHNvbiBqdXN0YW1lbnRlIGxvcwogICAgZG9zIHBlcmlvZG9zIHF1ZSBsYSB0ZXNpcyBxdWllcmUgY29udHJhc3RhcjogbWV6Y2xhcmxvcyBib3JyYSBwYXJ0ZSBkZSBsYQogICAgZGlmZXJlbmNpYSBxdWUgc2UgYnVzY2EgbWVkaXIuCgogICAgQXF1aSBzZSBlc3RhbmRhcml6YSBVTkEgc29sYSB2ZXogeSBwb3IgYW5pbywgZGVzcHVlcyBkZSByZXN1bWlyLiBBc2kgbGEKICAgIG1lZGlhbmEsIGxhIGRlc3ZpYWNpb24geSBsYSB0ZW5kZW5jaWEgc2UgY2FsY3VsYW4gc29icmUgdW5pZGFkZXMgZmlzaWNhcwogICAgcmVhbGVzIChuVCwga20vcywgSykgeSBlbCBlc2NhbGFkbyBvY3VycmUgYWwgZmluYWwsIGRvbmRlIGhhY2UgZmFsdGEgcGFyYQogICAgcXVlIEtNZWFucyBubyBsZSBkZSBtYXMgcGVzbyBhIGxhcyB2YXJpYWJsZXMgZGUgcmFuZ28gZ3JhbmRlLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgpTVUZJSk9TID0gKCJtZWRpYW4iLCAic3RkIiwgInRyZW5kIikKCgpkZWYgc3VtbWFyaXplX3dpbmRvd3Mod2luZG93czogbnAubmRhcnJheSwgKiwgaWdub3Jhcl9uYW46IGJvb2wgPSBGYWxzZSkgLT4gbnAubmRhcnJheToKICAgICIiIlJlc3VtZSBjYWRhIHZlbnRhbmEgYSAzIG51bWVyb3MgcG9yIHZhcmlhYmxlLgoKICAgIHdpbmRvd3M6IChucywgd3NpemUsIG5mKSAgLT4gIGRldnVlbHZlIChucywgbmYgKiAzKQoKICAgIENvbHVtbmFzIGVuIGVsIG9yZGVuIFttZWRfdjEsIHN0ZF92MSwgdHJlbmRfdjEsIG1lZF92Miwgc3RkX3YyLCAuLi5dLgoKICAgIExhIHRlbmRlbmNpYSBlcyBsYSBwZW5kaWVudGUgZGUgbGEgcmVjdGEgZGUgbWluaW1vcyBjdWFkcmFkb3Mgc29icmUgZWwgZWplCiAgICB0ZW1wb3JhbC4gQ29tbyB4ID0gYXJhbmdlKHdzaXplKSBlcyBmaWpvLCB0aWVuZSBmb3JtdWxhIGNlcnJhZGEgeSBzZSBjYWxjdWxhCiAgICBzb2JyZSB0b2RvIGVsIHRlbnNvciBkZSB1bmEgdmV6LCBlbiB2ZXogZGUgbGxhbWFyIGEgc2NpcHkuc3RhdHMubGlucmVncmVzcwogICAgdW5hIHZleiBwb3IgdmVudGFuYSB5IHZhcmlhYmxlIGNvbW8gaGFjaWEgZWwgb3JpZ2luYWwuCgogICAgYGlnbm9yYXJfbmFuPVRydWVgIGNhbGN1bGEgbG9zIHRyZXMgZXN0YWRpc3RpY29zIHNvYnJlIGxvcyBwdW50b3MgdmFsaWRvcywKICAgIHNpbiBuZWNlc2lkYWQgZGUgaW50ZXJwb2xhci4gRXMgbG8gY29oZXJlbnRlIGN1YW5kbyBlbCBkZXNjcmlwdG9yIGVzCiAgICBpbnRlZ3JhbDogdW5hIG1lZGlhbmEgc29icmUgMTUgZGUgMjAgcHVudG9zIGVzIHBlcmZlY3RhbWVudGUgdXRpbGl6YWJsZSwgeQogICAgYXNpIG5vIHNlIGludmVudGFuIHZhbG9yZXMgcXVlIGRlc3B1ZXMgZW50cmFuIGFsIG1vZGVsbyBjb21vIHNpIGZ1ZXJhbgogICAgbWVkaWRhcy4gQ29uIGxhIGFic2Npc2EgcmVzdHJpbmdpZGEgYSBsb3MgcHVudG9zIHZhbGlkb3MgbGEgcGVuZGllbnRlIHlhIG5vCiAgICB0aWVuZSB1bmEgZm9ybXVsYSBjb211biBhIHRvZGFzIGxhcyB2ZW50YW5hcywgZGUgbW9kbyBxdWUgbGEgbWVkaWEgZGUgeCBzZQogICAgcmVjYWxjdWxhIHBhcmEgY2FkYSB1bmEuCiAgICAiIiIKICAgIGlmIHdpbmRvd3MubmRpbSAhPSAzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJTZSBlc3BlcmFiYSB1biBhcnJheSAzRCAobnMsIHdzaXplLCBuZiksIGxsZWdvIHt3aW5kb3dzLnNoYXBlfSIpCgogICAgbnMsIHdzaXplLCBuZiA9IHdpbmRvd3Muc2hhcGUKICAgIGlmIHdzaXplIDwgMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJVbmEgdmVudGFuYSBuZWNlc2l0YSBhbCBtZW5vcyAyIHB1bnRvcyBwYXJhIHRlbmVyIHBlbmRpZW50ZSIpCgogICAgeCA9IG5wLmFyYW5nZSh3c2l6ZSwgZHR5cGU9bnAuZmxvYXQ2NCkKCiAgICBpZiBub3QgaWdub3Jhcl9uYW46CiAgICAgICAgeF9jZW50cmFkbyA9IHggLSB4Lm1lYW4oKQogICAgICAgIGRlbm9taW5hZG9yID0gKHhfY2VudHJhZG8qKjIpLnN1bSgpCiAgICAgICAgbWVkaWFuYSA9IG5wLm1lZGlhbih3aW5kb3dzLCBheGlzPTEpCiAgICAgICAgZGVzdmlhY2lvbiA9IG5wLnN0ZCh3aW5kb3dzLCBheGlzPTEpCiAgICAgICAgeV9tZWRpbyA9IHdpbmRvd3MubWVhbihheGlzPTEsIGtlZXBkaW1zPVRydWUpCiAgICAgICAgcGVuZGllbnRlID0gKAogICAgICAgICAgICAod2luZG93cyAtIHlfbWVkaW8pICogeF9jZW50cmFkb1tOb25lLCA6LCBOb25lXQogICAgICAgICkuc3VtKGF4aXM9MSkgLyBkZW5vbWluYWRvcgogICAgZWxzZToKICAgICAgICB2YWxpZG8gPSB+bnAuaXNuYW4od2luZG93cykKICAgICAgICBuID0gdmFsaWRvLnN1bShheGlzPTEpICAjIChucywgbmYpCgogICAgICAgIHdpdGggbnAuZXJyc3RhdGUoaW52YWxpZD0iaWdub3JlIiwgZGl2aWRlPSJpZ25vcmUiKToKICAgICAgICAgICAgbWVkaWFuYSA9IG5wLm5hbm1lZGlhbih3aW5kb3dzLCBheGlzPTEpCiAgICAgICAgICAgIGRlc3ZpYWNpb24gPSBucC5uYW5zdGQod2luZG93cywgYXhpcz0xKQoKICAgICAgICAgICAgIyBNZWRpYXMgcmVzdHJpbmdpZGFzIGEgbG9zIHB1bnRvcyB2YWxpZG9zIGRlIGNhZGEgdmVudGFuYS4KICAgICAgICAgICAgeF9tZWRpbyA9ICh4W05vbmUsIDosIE5vbmVdICogdmFsaWRvKS5zdW0oYXhpcz0xKSAvIG4KICAgICAgICAgICAgeV9tZWRpbyA9IG5wLm5hbnN1bSh3aW5kb3dzLCBheGlzPTEpIC8gbgoKICAgICAgICAgICAgZHggPSAoeFtOb25lLCA6LCBOb25lXSAtIHhfbWVkaW9bOiwgTm9uZSwgOl0pICogdmFsaWRvCiAgICAgICAgICAgIGR5ID0gbnAud2hlcmUodmFsaWRvLCB3aW5kb3dzIC0geV9tZWRpb1s6LCBOb25lLCA6XSwgMC4wKQogICAgICAgICAgICBkZW5vbWluYWRvciA9IChkeCoqMikuc3VtKGF4aXM9MSkKICAgICAgICAgICAgcGVuZGllbnRlID0gbnAud2hlcmUoZGVub21pbmFkb3IgPiAwLCAoZHggKiBkeSkuc3VtKGF4aXM9MSkgLyBkZW5vbWluYWRvciwgMC4wKQoKICAgICAgICAjIENvbiBtZW5vcyBkZSBkb3MgcHVudG9zIG5vIGhheSBwZW5kaWVudGUgZGVmaW5pZGEuCiAgICAgICAgcGVuZGllbnRlID0gbnAud2hlcmUobiA+PSAyLCBwZW5kaWVudGUsIDAuMCkKCiAgICAjIChucywgbmYsIDMpIC0+IChucywgbmYqMyksIGludGVyY2FsYW5kbyBsb3MgdHJlcyBlc3RhZGlzdGljb3MgcG9yIHZhcmlhYmxlLgogICAgcmV0dXJuIG5wLnN0YWNrKFttZWRpYW5hLCBkZXN2aWFjaW9uLCBwZW5kaWVudGVdLCBheGlzPTIpLnJlc2hhcGUobnMsIG5mICogMykKCgpkZWYgc3VtbWFyeV9jb2x1bW5zKHZhcmlhYmxlczogbGlzdFtzdHJdKSAtPiBsaXN0W3N0cl06CiAgICAiIiJOb21icmVzIGRlIGNvbHVtbmEgcXVlIHByb2R1Y2Ugc3VtbWFyaXplX3dpbmRvd3MsIGVuIGVsIG1pc21vIG9yZGVuLiIiIgogICAgcmV0dXJuIFtmInt2YXJ9X3tzdWZ9IiBmb3IgdmFyIGluIHZhcmlhYmxlcyBmb3Igc3VmIGluIFNVRklKT1NdCgoKZGVmIGJ1aWxkX3N1bW1hcnlfZnJhbWUoCiAgICB3aW5kb3dzOiBucC5uZGFycmF5LAogICAgc3RhcnRzOiBucC5uZGFycmF5LAogICAgdGllbXBvczogcGQuU2VyaWVzLAogICAgdmFyaWFibGVzOiBsaXN0W3N0cl0sCiAgICAqLAogICAgd2luZG93X3NpemU6IGludCwKICAgIGlnbm9yYXJfbmFuOiBib29sID0gRmFsc2UsCikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiRGF0YUZyYW1lIGNvbiBsYXMgZmVhdHVyZXMgcmVzdW1pZGFzIG1hcyBlbCBpbmljaW8geSBmaW4gZGUgY2FkYSB2ZW50YW5hLgoKICAgIGBzdGFydHNgIHNvbiBpbmRpY2VzIHBvc2ljaW9uYWxlcyBlbiBsYSBzZXJpZSBvcmlnaW5hbCAobG9zIHF1ZSBkZXZ1ZWx2ZQogICAgYG1ha2Vfd2luZG93c2ApLiBFbCBmaW4gZXMgZWwgdGllbXBvIGRlbCB1bHRpbW8gcHVudG8gZGUgbGEgdmVudGFuYS4KCiAgICBgaWdub3Jhcl9uYW49VHJ1ZWAgZXZpdGEgdGVuZXIgcXVlIGludGVycG9sYXIgYW50ZXM6IHZlciBgc3VtbWFyaXplX3dpbmRvd3NgLgogICAgIiIiCiAgICByZXN1bWVuID0gc3VtbWFyaXplX3dpbmRvd3Mod2luZG93cywgaWdub3Jhcl9uYW49aWdub3Jhcl9uYW4pCiAgICBkZiA9IHBkLkRhdGFGcmFtZShyZXN1bWVuLCBjb2x1bW5zPXN1bW1hcnlfY29sdW1ucyh2YXJpYWJsZXMpKQoKICAgIHQgPSBwZC50b19kYXRldGltZShwZC5TZXJpZXModGllbXBvcykucmVzZXRfaW5kZXgoZHJvcD1UcnVlKSkKICAgIGRmWyJzdGFydCJdID0gdC50b19udW1weSgpW3N0YXJ0c10KICAgIGRmWyJlbmQiXSA9IHQudG9fbnVtcHkoKVtzdGFydHMgKyB3aW5kb3dfc2l6ZSAtIDFdCiAgICByZXR1cm4gZGYKCgpkZWYgc3RhbmRhcmRpemVfYnlfeWVhcigKICAgIGRmOiBwZC5EYXRhRnJhbWUsICosIGNvbHVtbmFfdGllbXBvOiBzdHIgPSAic3RhcnQiLCBmZWF0dXJlczogbGlzdFtzdHJdIHwgTm9uZSA9IE5vbmUKKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJFc3RhbmRhcml6YSBsYXMgZmVhdHVyZXMgZGVudHJvIGRlIGNhZGEgYW5pbywgcG9yIHNlcGFyYWRvLgoKICAgIFNlIGhhY2UgcG9yIGFuaW8gYSBwcm9wb3NpdG86IDIwMDggKG1pbmltbyBzb2xhcikgeSAyMDE1IChtYXhpbW8gc29sYXIpCiAgICB0aWVuZW4gcmVnaW1lbmVzIGRpc3RpbnRvcy4gVW4gZXNjYWxhZG8gZ2xvYmFsIGRlamFyaWEgcXVlIGxhIGRpZmVyZW5jaWEKICAgIGVudHJlIGFtYm9zIHBlcmlvZG9zIGRvbWluZSB0b2RhcyBsYXMgY29tcG9uZW50ZXMgeSBLTWVhbnMgdGVybWluYXJpYQogICAgc2VwYXJhbmRvICJhbmlvIiBlbiB2ZXogZGUgInRpcG8gZGUgZXZlbnRvIi4KCiAgICBFcyBsYSBVTklDQSBlc3RhbmRhcml6YWNpb24gZGVsIHBpcGVsaW5lLiBObyB2b2x2ZXIgYSBlc2NhbGFyIGFudGVzIGRlCiAgICBhZ3J1cGFyOiBlc2UgZXJhIGVsIGJ1ZyAzLgogICAgIiIiCiAgICBpZiBmZWF0dXJlcyBpcyBOb25lOgogICAgICAgIGZlYXR1cmVzID0gW2MgZm9yIGMgaW4gZGYuY29sdW1ucyBpZiBjLmVuZHN3aXRoKFNVRklKT1MpXQoKICAgIG91dCA9IGRmLmNvcHkoKQogICAgb3V0W2NvbHVtbmFfdGllbXBvXSA9IHBkLnRvX2RhdGV0aW1lKG91dFtjb2x1bW5hX3RpZW1wb10pCiAgICBhbmlvcyA9IG91dFtjb2x1bW5hX3RpZW1wb10uZHQueWVhcgoKICAgIGZvciBfYW5pbywgaWR4IGluIG91dC5ncm91cGJ5KGFuaW9zKS5ncm91cHMuaXRlbXMoKToKICAgICAgICBibG9xdWUgPSBvdXQubG9jW2lkeCwgZmVhdHVyZXNdCiAgICAgICAgbWVkaWEgPSBibG9xdWUubWVhbigpCiAgICAgICAgZGVzdiA9IGJsb3F1ZS5zdGQoZGRvZj0wKQogICAgICAgICMgVW5hIGZlYXR1cmUgY29uc3RhbnRlIGVuIGVsIGFuaW8gdGllbmUgZGVzdmlhY2lvbiAwOiBzZSBkZWphIGVuIDAgZW4KICAgICAgICAjIHZleiBkZSBwcm9kdWNpciBpbmYgbyBOYU4gYWwgZGl2aWRpci4KICAgICAgICBkZXN2ID0gZGVzdi5yZXBsYWNlKDAsIG5wLm5hbikKICAgICAgICBvdXQubG9jW2lkeCwgZmVhdHVyZXNdID0gKChibG9xdWUgLSBtZWRpYSkgLyBkZXN2KS5maWxsbmEoMC4wKQoKICAgIHJldHVybiBvdXQK",
    "tesis/features/nativo.py": "IiIiVmVudGFuYXMgcG9yIGR1cmFjaW9uLCBzb2JyZSBsYSByZXNvbHVjaW9uIG5hdGl2YSBkZSBjYWRhIGluc3RydW1lbnRvLgoKRWwgcGlwZWxpbmUgY2xhc2ljbyBwcm9tZWRpYSB0b2RvIGEgNCBtaW51dG9zIHkgZGVzcHVlcyBjb3J0YSB2ZW50YW5hcyBkZSAyMApwdW50b3MuIEVzbyB0aWVuZSB1biBjb3N0ZSBxdWUgbm8gc2UgdmU6IGFsIHByb21lZGlhciBhIDQgbWluIGxhIHZhcmlhYmlsaWRhZApyYXBpZGEgZGVzYXBhcmVjZSBBTlRFUyBkZSBtZWRpcmxhLCBkZSBtb2RvIHF1ZSBsYSBkZXN2aWFjaW9uIGRlIGxhIHZlbnRhbmEgZXMKbGEgZGVzdmlhY2lvbiBkZSB2ZWludGUgcHJvbWVkaW9zLCBubyBsYSBkZSBsb3MgZGF0b3MuIFBhcmEgZXN0cnVjdHVyYXMgY3V5YQpmaXJtYSBlcyB0dXJidWxlbnRhIC1sYXMgU0lSLCBzb2JyZSB0b2RvLSBlc28gcHVlZGUgZXN0YXIgYm9ycmFuZG8ganVzdG8gbGEKc2VuYWwgcXVlIHNlIGJ1c2NhLgoKQXF1aSBsYSB2ZW50YW5hIHNlIGRlZmluZSBwb3IgRFVSQUNJT04gKDgwIG1pbnV0b3MpIHkgY2FkYSB2YXJpYWJsZSBhcG9ydGEgc3VzCnByb3Bpb3MgcHVudG9zIG5hdGl2b3M6IFNXRVBBTSBhIH42NCBzLCBNQUcgYSB+MSBtaW4uIE5vIHNlIHByb21lZGlhIG5hZGEgYW50ZXMKZGUgcmVzdW1pci4KCkVsIGNhbGN1bG8gc2UgYXBveWEgZW4gbGFzIHZlbnRhbmFzIHRlbXBvcmFsZXMgZGUgcGFuZGFzIChgcm9sbGluZygnODBtaW4nKWApLApxdWUgZXN0YW4gaW1wbGVtZW50YWRhcyBlbiBDLiBNZWRpYW5hIHkgZGVzdmlhY2lvbiBzYWxlbiBkaXJlY3RhczsgcGFyYSBsYQpwZW5kaWVudGUgc2UgdXNhbiBzdW1hcyBhY3VtdWxhZGFzLCBxdWUgZGFuIGxhIGZvcm11bGEgZGUgbWluaW1vcyBjdWFkcmFkb3Mgc2luCnRlbmVyIHF1ZSByZWNvcnJlciB2ZW50YW5hIHBvciB2ZW50YW5hOgoKICAgIG0gPSAobipTeHkgLSBTeCpTeSkgLyAobipTeHggLSBTeF4yKQoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgpTVUZJSk9TID0gKCJtZWRpYW4iLCAic3RkIiwgInRyZW5kIikKCgpkZWYgX2VzdGFkaXN0aWNvc19yb2xsaW5nKAogICAgZGY6IHBkLkRhdGFGcmFtZSwKICAgIHZhcmlhYmxlczogbGlzdFtzdHJdLAogICAgKiwKICAgIGR1cmFjaW9uOiBzdHIsCiAgICBjb2x1bW5hX3RpZW1wbzogc3RyID0gIlRpZW1wbyIsCikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiTWVkaWFuYSwgZGVzdmlhY2lvbiwgcGVuZGllbnRlIHkgbnVtZXJvIGRlIHB1bnRvcyBlbiBjYWRhIHZlbnRhbmEgbW92aWwuCgogICAgTGEgdmVudGFuYSBlcyBjZXJyYWRhIHBvciBsYSBkZXJlY2hhOiBjYWRhIGZpbGEgcmVzdW1lIGxvcyBgZHVyYWNpb25gCiAgICBhbnRlcmlvcmVzIGhhc3RhIGVzZSBpbnN0YW50ZSBpbmNsdXNpdmUuCiAgICAiIiIKICAgIHMgPSBkZi5zZXRfaW5kZXgoY29sdW1uYV90aWVtcG8pLnNvcnRfaW5kZXgoKQoKICAgICMgRWwgdGllbXBvIGVuIHNlZ3VuZG9zLCBjZW50cmFkbyBlbiBlbCBwcmltZXIgaW5zdGFudGUgcGFyYSBubyBwZXJkZXIKICAgICMgcHJlY2lzaW9uIGFsIGVsZXZhcmxvIGFsIGN1YWRyYWRvLgogICAgdCA9IChzLmluZGV4IC0gcy5pbmRleFswXSkudG90YWxfc2Vjb25kcygpLnRvX251bXB5KCkKCiAgICBzYWxpZGEgPSB7fQogICAgZm9yIHYgaW4gdmFyaWFibGVzOgogICAgICAgIHkgPSBzW3ZdCiAgICAgICAgdmFsaWRvID0geS5ub3RuYSgpCiAgICAgICAgdHYgPSBwZC5TZXJpZXMobnAud2hlcmUodmFsaWRvLCB0LCBucC5uYW4pLCBpbmRleD1zLmluZGV4KQoKICAgICAgICByX3kgPSB5LnJvbGxpbmcoZHVyYWNpb24pCiAgICAgICAgbiA9IHJfeS5jb3VudCgpCgogICAgICAgIHN4ID0gdHYucm9sbGluZyhkdXJhY2lvbikuc3VtKCkKICAgICAgICBzeSA9IHJfeS5zdW0oKQogICAgICAgIHN4eSA9ICh0diAqIHkpLnJvbGxpbmcoZHVyYWNpb24pLnN1bSgpCiAgICAgICAgc3h4ID0gKHR2ICogdHYpLnJvbGxpbmcoZHVyYWNpb24pLnN1bSgpCgogICAgICAgIGRlbm9taW5hZG9yID0gbiAqIHN4eCAtIHN4KioyCiAgICAgICAgd2l0aCBucC5lcnJzdGF0ZShpbnZhbGlkPSJpZ25vcmUiLCBkaXZpZGU9Imlnbm9yZSIpOgogICAgICAgICAgICBwZW5kaWVudGUgPSAobiAqIHN4eSAtIHN4ICogc3kpIC8gZGVub21pbmFkb3IKICAgICAgICAjIFNpbiBhbCBtZW5vcyBkb3MgcHVudG9zIGRpc3RpbnRvcyBsYSBwZW5kaWVudGUgbm8gZXN0YSBkZWZpbmlkYS4KICAgICAgICBwZW5kaWVudGUgPSBwZW5kaWVudGUud2hlcmUoKG4gPj0gMikgJiAoZGVub21pbmFkb3IgPiAwKSwgMC4wKQoKICAgICAgICBzYWxpZGFbZiJ7dn1fbWVkaWFuIl0gPSByX3kubWVkaWFuKCkKICAgICAgICBzYWxpZGFbZiJ7dn1fc3RkIl0gPSByX3kuc3RkKGRkb2Y9MCkKICAgICAgICAjIERlIHVuaWRhZGVzIHBvciBzZWd1bmRvIGEgdW5pZGFkZXMgcG9yIG1pbnV0bywgbWFzIGxlZ2libGUuCiAgICAgICAgc2FsaWRhW2Yie3Z9X3RyZW5kIl0gPSBwZW5kaWVudGUgKiA2MC4wCiAgICAgICAgc2FsaWRhW2YiX19uX3t2fSJdID0gbgoKICAgIHJldHVybiBwZC5EYXRhRnJhbWUoc2FsaWRhLCBpbmRleD1zLmluZGV4KQoKCmRlZiByZXN1bWlyX25hdGl2bygKICAgIGZ1ZW50ZXM6IGRpY3Rbc3RyLCB0dXBsZVtwZC5EYXRhRnJhbWUsIGxpc3Rbc3RyXV1dLAogICAgKiwKICAgIGR1cmFjaW9uOiBzdHIgPSAiODBtaW4iLAogICAgcGFzbzogc3RyID0gIjRtaW4iLAogICAgbWluaW1vX3ZhbGlkbzogZmxvYXQgPSAwLjc1LAogICAgY29sdW1uYV90aWVtcG86IHN0ciA9ICJUaWVtcG8iLAopIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIlJlc3VtZSB2ZW50YW5hcyBkZSBgZHVyYWNpb25gIHVzYW5kbyBsYSByZXNvbHVjaW9uIG5hdGl2YSBkZSBjYWRhIGZ1ZW50ZS4KCiAgICBmdWVudGVzOiB7bm9tYnJlOiAoZGF0YWZyYW1lLCB2YXJpYWJsZXMpfS4gQ2FkYSBkYXRhZnJhbWUgY29uc2VydmEgc3UgcHJvcGlhCiAgICBjYWRlbmNpYTsgbm8gc2UgcHJvbWVkaWFuIGVudHJlIHNpLgoKICAgIERldnVlbHZlIHVuIERhdGFGcmFtZSBjb24gYHN0YXJ0YCwgYGVuZGAgeSB0cmVzIGNvbHVtbmFzIHBvciB2YXJpYWJsZS4gU2UKICAgIGNvbnNlcnZhIHVuYSB2ZW50YW5hIHNvbG8gc2kgVE9EQVMgc3VzIHZhcmlhYmxlcyBhbGNhbnphbiBgbWluaW1vX3ZhbGlkb2AKICAgIGRlIGxvcyBwdW50b3MgcXVlIGNhYnJpYW4gc2VndW4gbGEgY2FkZW5jaWEgbmF0aXZhIGRlIHN1IGZ1ZW50ZS4KICAgICIiIgogICAgZHVyID0gcGQuVGltZWRlbHRhKGR1cmFjaW9uKQogICAgcGFydGVzLCBleGlnaWRvcyA9IFtdLCB7fQoKICAgIGZvciBfbm9tYnJlLCAoZGYsIHZhcmlhYmxlcykgaW4gZnVlbnRlcy5pdGVtcygpOgogICAgICAgIGlmIGRmLmVtcHR5OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGVzdCA9IF9lc3RhZGlzdGljb3Nfcm9sbGluZyhkZiwgdmFyaWFibGVzLCBkdXJhY2lvbj1kdXJhY2lvbiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY29sdW1uYV90aWVtcG89Y29sdW1uYV90aWVtcG8pCiAgICAgICAgcGFydGVzLmFwcGVuZChlc3QpCgogICAgICAgICMgUHVudG9zIGVzcGVyYWRvcyBlbiB1bmEgdmVudGFuYSwgYSBwYXJ0aXIgZGUgbGEgY2FkZW5jaWEgb2JzZXJ2YWRhLgogICAgICAgIHQgPSBwZC50b19kYXRldGltZShkZltjb2x1bW5hX3RpZW1wb10pLnNvcnRfdmFsdWVzKCkKICAgICAgICBjYWRlbmNpYSA9IHQuZGlmZigpLm1lZGlhbigpCiAgICAgICAgZXNwZXJhZG9zID0gbWF4KDIsIGludChkdXIgLyBjYWRlbmNpYSkpCiAgICAgICAgZm9yIHYgaW4gdmFyaWFibGVzOgogICAgICAgICAgICBleGlnaWRvc1t2XSA9IGVzcGVyYWRvcyAqIG1pbmltb192YWxpZG8KCiAgICBpZiBub3QgcGFydGVzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk5vIHNlIHJlY2liaW8gbmluZ3VuYSBmdWVudGUgY29uIGRhdG9zIikKCiAgICAjIFJlamlsbGEgY29tdW46IGxvcyBpbnN0YW50ZXMgZW4gbG9zIHF1ZSBzZSBldmFsdWEgY2FkYSB2ZW50YW5hLgogICAgaW5pY2lvID0gbWF4KHAuaW5kZXhbMF0gZm9yIHAgaW4gcGFydGVzKSArIGR1cgogICAgZmluID0gbWluKHAuaW5kZXhbLTFdIGZvciBwIGluIHBhcnRlcykKICAgIGlmIGluaWNpbyA+PSBmaW46CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTGFzIGZ1ZW50ZXMgbm8gc2Ugc29sYXBhbiBsbyBzdWZpY2llbnRlIikKICAgIHJlamlsbGEgPSBwZC5kYXRlX3JhbmdlKGluaWNpbywgZmluLCBmcmVxPXBhc28pCgogICAgIyBDYWRhIGZ1ZW50ZSBzZSBtdWVzdHJlYSBlbiBsYSByZWppbGxhIGNvbiBlbCB1bHRpbW8gdmFsb3IgZGlzcG9uaWJsZS4KICAgIGFsaW5lYWRhcyA9IFtwLnJlaW5kZXgocmVqaWxsYSwgbWV0aG9kPSJmZmlsbCIsIHRvbGVyYW5jZT1kdXIpIGZvciBwIGluIHBhcnRlc10KICAgIGp1bnRhcyA9IHBkLmNvbmNhdChhbGluZWFkYXMsIGF4aXM9MSkKCiAgICAjIEZpbHRybyBkZSBjb2JlcnR1cmE6IGNhZGEgdmFyaWFibGUgY29uIHN1cyBwcm9waW9zIHB1bnRvcyBlc3BlcmFkb3MuCiAgICBzdWZpY2llbnRlID0gcGQuU2VyaWVzKFRydWUsIGluZGV4PWp1bnRhcy5pbmRleCkKICAgIGZvciB2LCBtaW5pbW8gaW4gZXhpZ2lkb3MuaXRlbXMoKToKICAgICAgICBzdWZpY2llbnRlICY9IGp1bnRhc1tmIl9fbl97dn0iXSA+PSBtaW5pbW8KICAgIGp1bnRhcyA9IGp1bnRhc1tzdWZpY2llbnRlXQoKICAgIGp1bnRhcyA9IGp1bnRhcy5kcm9wKGNvbHVtbnM9W2MgZm9yIGMgaW4ganVudGFzLmNvbHVtbnMgaWYgYy5zdGFydHN3aXRoKCJfX25fIildKQogICAganVudGFzID0ganVudGFzLmRyb3BuYSgpCgogICAganVudGFzID0ganVudGFzLnJlc2V0X2luZGV4KG5hbWVzPSJlbmQiKQogICAganVudGFzWyJzdGFydCJdID0ganVudGFzWyJlbmQiXSAtIGR1cgogICAgb3JkZW4gPSBbYyBmb3IgYyBpbiBqdW50YXMuY29sdW1ucyBpZiBjIG5vdCBpbiAoInN0YXJ0IiwgImVuZCIpXQogICAgcmV0dXJuIGp1bnRhc1tbKm9yZGVuLCAic3RhcnQiLCAiZW5kIl1dCgoKZGVmIGNvbHVtbmFzX3Jlc3VtZW4odmFyaWFibGVzOiBsaXN0W3N0cl0pIC0+IGxpc3Rbc3RyXToKICAgICIiIk5vbWJyZXMgZGUgY29sdW1uYSBxdWUgcHJvZHVjZSBgcmVzdW1pcl9uYXRpdm9gLCBlbiBlbCBtaXNtbyBvcmRlbi4iIiIKICAgIHJldHVybiBbZiJ7dn1fe3N9IiBmb3IgdiBpbiB2YXJpYWJsZXMgZm9yIHMgaW4gU1VGSUpPU10K",
    "tesis/models/labeling.py": "IiIiRXRpcXVldGFkbyBkZSB2ZW50YW5hcyBjb250cmEgbG9zIGNhdGFsb2dvcyBkZSBJQ01FIHkgU0lSLgoKVW5hIHZlbnRhbmEgc2UgZXRpcXVldGEgc2VndW4gY3VhbnRvIGRlIHN1IGR1cmFjaW9uIGNhZSBkZW50cm8gZGUgdW4gZXZlbnRvCmNhdGFsb2dhZG86CgogICAgSUNNRSAgICAgIHNvbGFwYSA+PSB1bWJyYWwgY29uIHVuIGV2ZW50byBJQ01FICAgKHRpZW5lIHByaW9yaWRhZCkKICAgIFNJUiAgICAgICBzb2xhcGEgPj0gdW1icmFsIGNvbiB1biBldmVudG8gU0lSCiAgICBGb25kbyAgICAgbm8gc29sYXBhIG5hZGEKICAgIEZyb250ZXJhICBzb2xhcGEgYWxnbywgcGVybyBwb3IgZGViYWpvIGRlbCB1bWJyYWwgLT4gc2UgZGVzY2FydGEKClJFTkRJTUlFTlRPIC0gRWwgb3JpZ2luYWwgcmVjb3JyaWEgZWwgY2F0YWxvZ28gZW50ZXJvIGNvbiAuaXRlcnJvd3MoKSBwb3IgY2FkYQogICAgdmVudGFuYSwgZGVudHJvIGRlIHVuIGRmLmFwcGx5KCkuIENvbiBkZWNlbmFzIGRlIG1pbGVzIGRlIHZlbnRhbmFzIHkgfjEwMAogICAgZXZlbnRvcyBzb24gbWlsbG9uZXMgZGUgaXRlcmFjaW9uZXMgZW4gUHl0aG9uIHB1cm8uIEFxdWkgZWwgc29sYXBhbWllbnRvIHNlCiAgICBjYWxjdWxhIHBvciBicm9hZGNhc3RpbmcgZGUgTnVtUHkgc29icmUgbGEgbWF0cml6ICh2ZW50YW5hcyB4IGV2ZW50b3MpLgoKWWEgbm8gZXhpc3RlIGxhIGV0aXF1ZXRhICdVbmtub3duX1llYXInIGRlbCBvcmlnaW5hbDogZXJhIGNvbnNlY3VlbmNpYSBkZWwgYnVnIDQKKHZlbnRhbmFzIHF1ZSBjcnV6YWJhbiBlbCBodWVjbyAyMDA4LTIwMTUpIHkgZGVzYXBhcmVjZSBhbCBnZW5lcmFyIGxhcyB2ZW50YW5hcwpwb3IgdHJhbW8gY29udGludW8uIExvcyBjYXRhbG9nb3MgdGFtcG9jbyBzZSBwYXJ0ZW4gcG9yIGFuaW86IGVsIHNvbGFwYW1pZW50bwp0ZW1wb3JhbCB5YSBkZWNpZGUgc29sYSBjdWFsIGV2ZW50byBhcGxpY2EuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCklDTUUgPSAiSUNNRSIKU0lSID0gIlNJUiIKRk9ORE8gPSAiRm9uZG8iCkZST05URVJBID0gIkZyb250ZXJhIgoKCmRlZiBvdmVybGFwX3JhdGlvKAogICAgc3RhcnRzOiBwZC5TZXJpZXMsIGVuZHM6IHBkLlNlcmllcywgZXZlbnRvczogcGQuRGF0YUZyYW1lCikgLT4gbnAubmRhcnJheToKICAgICIiIkZyYWNjaW9uIGRlIGNhZGEgdmVudGFuYSBjdWJpZXJ0YSBwb3IgZWwgZXZlbnRvIHF1ZSBtYXMgbGEgc29sYXBhLgoKICAgIHN0YXJ0cy9lbmRzOiAobl92ZW50YW5hcywpIGRhdGV0aW1lCiAgICBldmVudG9zOiAgICAgRGF0YUZyYW1lIGNvbiBjb2x1bW5hcyAnc3RhcnRfdGltZScgeSAnZW5kX3RpbWUnCiAgICBkZXZ1ZWx2ZTogICAgKG5fdmVudGFuYXMsKSBmbG9hdCBlbiBbMCwgMV0KCiAgICBTaSBlbCBjYXRhbG9nbyBlc3RhIHZhY2lvIGRldnVlbHZlIGNlcm9zLgogICAgIiIiCiAgICB3X2luaSA9IHBkLnRvX2RhdGV0aW1lKHBkLlNlcmllcyhzdGFydHMpKS50b19udW1weSgiZGF0ZXRpbWU2NFtuc10iKS5hc3R5cGUoImludDY0IikKICAgIHdfZmluID0gcGQudG9fZGF0ZXRpbWUocGQuU2VyaWVzKGVuZHMpKS50b19udW1weSgiZGF0ZXRpbWU2NFtuc10iKS5hc3R5cGUoImludDY0IikKCiAgICBpZiBldmVudG9zIGlzIE5vbmUgb3IgbGVuKGV2ZW50b3MpID09IDA6CiAgICAgICAgcmV0dXJuIG5wLnplcm9zKGxlbih3X2luaSksIGR0eXBlPW5wLmZsb2F0NjQpCgogICAgZV9pbmkgPSAoCiAgICAgICAgcGQudG9fZGF0ZXRpbWUoZXZlbnRvc1sic3RhcnRfdGltZSJdKS50b19udW1weSgiZGF0ZXRpbWU2NFtuc10iKS5hc3R5cGUoImludDY0IikKICAgICkKICAgIGVfZmluID0gcGQudG9fZGF0ZXRpbWUoZXZlbnRvc1siZW5kX3RpbWUiXSkudG9fbnVtcHkoImRhdGV0aW1lNjRbbnNdIikuYXN0eXBlKCJpbnQ2NCIpCgogICAgZHVyYWNpb24gPSAod19maW4gLSB3X2luaSkuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICAjIFVuYSB2ZW50YW5hIGRlIGR1cmFjaW9uIGNlcm8gbm8gcHVlZGUgc29sYXBhciBuYWRhOiBzZSBldml0YSBkaXZpZGlyIHBvciAwLgogICAgZHVyYWNpb25bZHVyYWNpb24gPT0gMF0gPSBucC5uYW4KCiAgICAjIChuX3ZlbnRhbmFzLCBuX2V2ZW50b3MpIHBvciBicm9hZGNhc3RpbmcuCiAgICBpbmljaW8gPSBucC5tYXhpbXVtKHdfaW5pWzosIE5vbmVdLCBlX2luaVtOb25lLCA6XSkKICAgIGZpbiA9IG5wLm1pbmltdW0od19maW5bOiwgTm9uZV0sIGVfZmluW05vbmUsIDpdKQogICAgc29sYXBlID0gbnAuY2xpcChmaW4gLSBpbmljaW8sIDAsIE5vbmUpLmFzdHlwZShucC5mbG9hdDY0KQoKICAgIG1lam9yID0gc29sYXBlLm1heChheGlzPTEpIC8gZHVyYWNpb24KICAgIHJldHVybiBucC5uYW5fdG9fbnVtKG1lam9yLCBuYW49MC4wKQoKCmRlZiBsYWJlbF93aW5kb3dzKAogICAgZGY6IHBkLkRhdGFGcmFtZSwKICAgIGljbWU6IHBkLkRhdGFGcmFtZSwKICAgIHNpcjogcGQuRGF0YUZyYW1lLAogICAgKiwKICAgIHVtYnJhbDogZmxvYXQgPSAwLjgsCiAgICBjb2xfaW5pY2lvOiBzdHIgPSAic3RhcnQiLAogICAgY29sX2Zpbjogc3RyID0gImVuZCIsCikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiQWdyZWdhIGxhcyBjb2x1bW5hcyAnZXZlbnRfbGFiZWwnLCAnaWNtZV9vdmVybGFwJyB5ICdzaXJfb3ZlcmxhcCcuCgogICAgU2UgZ3VhcmRhbiB0YW1iaWVuIGxvcyBzb2xhcGFtaWVudG9zIGNydWRvcywgbm8gc29sbyBsYSBldGlxdWV0YTogcGVybWl0ZW4KICAgIGp1c3RpZmljYXIgZWwgdW1icmFsIGVuIGxhIHRlc2lzIHkgcmVoYWNlciBlbCBldGlxdWV0YWRvIGNvbiBvdHJvIHZhbG9yIHNpbgogICAgcmVjYWxjdWxhciBuYWRhLgogICAgIiIiCiAgICBpZiBub3QgMCA8IHVtYnJhbCA8PSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFbCB1bWJyYWwgZGViZSBlc3RhciBlbiAoMCwgMV0sIGxsZWdvIHt1bWJyYWx9IikKCiAgICBvdXQgPSBkZi5jb3B5KCkKICAgIHJfaWNtZSA9IG92ZXJsYXBfcmF0aW8ob3V0W2NvbF9pbmljaW9dLCBvdXRbY29sX2Zpbl0sIGljbWUpCiAgICByX3NpciA9IG92ZXJsYXBfcmF0aW8ob3V0W2NvbF9pbmljaW9dLCBvdXRbY29sX2Zpbl0sIHNpcikKCiAgICBldGlxdWV0YXMgPSBucC5mdWxsKGxlbihvdXQpLCBGUk9OVEVSQSwgZHR5cGU9b2JqZWN0KQogICAgZXRpcXVldGFzWyhyX2ljbWUgPT0gMCkgJiAocl9zaXIgPT0gMCldID0gRk9ORE8KICAgIGV0aXF1ZXRhc1tyX3NpciA+PSB1bWJyYWxdID0gU0lSCiAgICBldGlxdWV0YXNbcl9pY21lID49IHVtYnJhbF0gPSBJQ01FICAjIHVsdGltbzogSUNNRSBtYW5kYSBzb2JyZSBTSVIKCiAgICBvdXRbImljbWVfb3ZlcmxhcCJdID0gcl9pY21lCiAgICBvdXRbInNpcl9vdmVybGFwIl0gPSByX3NpcgogICAgb3V0WyJldmVudF9sYWJlbCJdID0gZXRpcXVldGFzCiAgICByZXR1cm4gb3V0CgoKZGVmIGRyb3BfZnJvbnRlcmEoZGY6IHBkLkRhdGFGcmFtZSwgKiwgY29sdW1uYTogc3RyID0gImV2ZW50X2xhYmVsIikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiUXVpdGEgbGFzIHZlbnRhbmFzIGRlIHRyYW5zaWNpb24sIHF1ZSBubyBzb24gbmkgZXZlbnRvIG5pIGZvbmRvIGxpbXBpby4iIiIKICAgIHJldHVybiBkZltkZltjb2x1bW5hXSAhPSBGUk9OVEVSQV0uY29weSgpCg==",
    "tesis/models/clustering.py": "IiIiS01lYW5zIHNvYnJlIGxhcyB2ZW50YW5hcyByZXN1bWlkYXMgeSBzdSBldmFsdWFjaW9uIGNvbnRyYSBsb3MgY2F0YWxvZ29zLgoKQlVHIDIgLSBFbCBub3RlYm9vayBjb25zdHJ1aWEgZGZfd2luZG93c19sYWJlbGVkX3VuZGVyc2FtcGxlZCAoZG9zIHZlY2VzLCBlbiBsYXMKICAgIGNlbGRhcyA1NiB5IDU3LCBjb2RpZ28gZHVwbGljYWRvKSB5IGRlc3B1ZXMgY29ycmlhIGVsIEtNZWFucyBzb2JyZQogICAgZGZfd2luZG93c19sYWJlbGVkX2ZpbHRlcmVkLCBlbCBjb25qdW50byBERVNCQUxBTkNFQURPLiBUb2RvIGVsIHN1Ym11ZXN0cmVvCiAgICBlcmEgZGVjb3JhdGl2by4gRWwgbWFya2Rvd24gZGUgbGEgY2VsZGEgNjEgcHJvbWV0ZSBjb21wYXJhciBjb250cmEgbG9zIGRhdG9zCiAgICBzdWJtdWVzdHJlYWRvczsgZXNhIGNvbXBhcmFjaW9uIG51bmNhIG9jdXJyaWEuCgogICAgQXF1aSBlbCBzdWJtdWVzdHJlbyBlcyB1biBwYXJhbWV0cm8gZXhwbGljaXRvIGRlIGBydW5fa21lYW5zYC4gU2kgc2UgcGlkZSwKICAgIHNlIGFwbGljYSBkZSB2ZXJkYWQ7IHNpIG5vLCBubyBzZSBjYWxjdWxhLiBObyBoYXkgZm9ybWEgZGUgY3JlZXIgcXVlIHNlCiAgICBhcGxpY28gc2luIHF1ZSBzZSBoYXlhIGFwbGljYWRvLgoKVGFtcG9jbyBzZSB2dWVsdmUgYSBlc3RhbmRhcml6YXI6IGxhcyBmZWF0dXJlcyB5YSB2aWVuZW4gZXNjYWxhZGFzIHBvciBhbmlvCmRlc2RlIGZlYXR1cmVzL3N1bW1hcnkucHkuIEVzY2FsYXIgb3RyYSB2ZXogYXF1aSBlcmEgbGEgdGVyY2VyYSBwYXNhZGEgZGVsIGJ1ZyAzLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gc2tsZWFybi5jbHVzdGVyIGltcG9ydCBLTWVhbnMKZnJvbSBza2xlYXJuLmRlY29tcG9zaXRpb24gaW1wb3J0IFBDQQoKZnJvbSB0ZXNpcy5mZWF0dXJlcy5zdW1tYXJ5IGltcG9ydCBTVUZJSk9TCmZyb20gdGVzaXMubW9kZWxzLmxhYmVsaW5nIGltcG9ydCBGT05ETwoKU0VNSUxMQSA9IDQyCgoKZGVmIGZlYXR1cmVfbWF0cml4KGRmOiBwZC5EYXRhRnJhbWUpIC0+IHR1cGxlW25wLm5kYXJyYXksIGxpc3Rbc3RyXV06CiAgICAiIiJFeHRyYWUgbGFzIGNvbHVtbmFzIGRlIGZlYXR1cmVzIChtZWRpYW4vc3RkL3RyZW5kKSBjb21vIG1hdHJpei4iIiIKICAgIGNvbHMgPSBbYyBmb3IgYyBpbiBkZi5jb2x1bW5zIGlmIGMuZW5kc3dpdGgoU1VGSUpPUyldCiAgICBpZiBub3QgY29sczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJObyBzZSBlbmNvbnRyYXJvbiBjb2x1bW5hcyBkZSBmZWF0dXJlcyBlbiBlbCBEYXRhRnJhbWUiKQogICAgcmV0dXJuIGRmW2NvbHNdLnRvX251bXB5KGR0eXBlPW5wLmZsb2F0NjQpLCBjb2xzCgoKZGVmIHVuZGVyc2FtcGxlX2ZvbmRvKAogICAgZGY6IHBkLkRhdGFGcmFtZSwKICAgICosCiAgICBjb2x1bW5hOiBzdHIgPSAiZXZlbnRfbGFiZWwiLAogICAgcmFuZG9tX3N0YXRlOiBpbnQgPSBTRU1JTExBLAopIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIlJlZHVjZSBsYSBjbGFzZSAnRm9uZG8nIGFsIHRhbWFubyBkZSBsYSBzZWd1bmRhIGNsYXNlIG1hcyBncmFuZGUuCgogICAgJ0ZvbmRvJyBkb21pbmEgcG9yIGNvbnN0cnVjY2lvbjogbGEgbWF5b3IgcGFydGUgZGVsIHRpZW1wbyBlbCB2aWVudG8gc29sYXIKICAgIGVzdGEgdHJhbnF1aWxvLiBTaW4gZXF1aWxpYnJhciwgS01lYW5zIGRlZGljYSBjYXNpIHRvZG9zIGxvcyBjZW50cm9pZGVzIGEKICAgIGRlc2NyaWJpciBlbCBmb25kbyB5IGxvcyBldmVudG9zIHF1ZWRhbiByZXBhcnRpZG9zIGNvbW8gcnVpZG8uCiAgICAiIiIKICAgIGNvbnRlb3MgPSBkZltjb2x1bW5hXS52YWx1ZV9jb3VudHMoKQogICAgaWYgRk9ORE8gbm90IGluIGNvbnRlb3MuaW5kZXg6CiAgICAgICAgcmV0dXJuIGRmLmNvcHkoKQoKICAgIG90cmFzID0gY29udGVvcy5kcm9wKEZPTkRPKQogICAgaWYgb3RyYXMuZW1wdHk6CiAgICAgICAgcmV0dXJuIGRmLmNvcHkoKQoKICAgIG9iamV0aXZvID0gaW50KG90cmFzLm1heCgpKQogICAgZm9uZG8gPSBkZltkZltjb2x1bW5hXSA9PSBGT05ET10KICAgIGlmIGxlbihmb25kbykgPD0gb2JqZXRpdm86CiAgICAgICAgcmV0dXJuIGRmLmNvcHkoKQoKICAgIHJlc3RvID0gZGZbZGZbY29sdW1uYV0gIT0gRk9ORE9dCiAgICBmb25kb19yZWR1Y2lkbyA9IGZvbmRvLnNhbXBsZShuPW9iamV0aXZvLCByYW5kb21fc3RhdGU9cmFuZG9tX3N0YXRlKQogICAgcmV0dXJuICgKICAgICAgICBwZC5jb25jYXQoW2ZvbmRvX3JlZHVjaWRvLCByZXN0b10pCiAgICAgICAgLnNhbXBsZShmcmFjPTEsIHJhbmRvbV9zdGF0ZT1yYW5kb21fc3RhdGUpCiAgICAgICAgLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgICkKCgpkZWYgcnVuX2ttZWFucygKICAgIGRmOiBwZC5EYXRhRnJhbWUsCiAgICBrOiBpbnQsCiAgICAqLAogICAgdW5kZXJzYW1wbGU6IGJvb2wgPSBGYWxzZSwKICAgIHJhbmRvbV9zdGF0ZTogaW50ID0gU0VNSUxMQSwKKSAtPiB0dXBsZVtwZC5EYXRhRnJhbWUsIEtNZWFuc106CiAgICAiIiJBZ3J1cGEgbGFzIHZlbnRhbmFzIHkgZGV2dWVsdmUgZWwgRGF0YUZyYW1lIGNvbiBsYSBjb2x1bW5hICdjbHVzdGVyJy4KCiAgICBgdW5kZXJzYW1wbGU9VHJ1ZWAgZXF1aWxpYnJhIGxhIGNsYXNlICdGb25kbycgQU5URVMgZGUgYWdydXBhci4gRXMgZXhwbGljaXRvCiAgICBqdXN0YW1lbnRlIHBvcnF1ZSBlbCBvcmlnaW5hbCBkZWNpYSBoYWNlcmxvIHkgbm8gbG8gaGFjaWEuCiAgICAiIiIKICAgIGRhdG9zID0gdW5kZXJzYW1wbGVfZm9uZG8oZGYsIHJhbmRvbV9zdGF0ZT1yYW5kb21fc3RhdGUpIGlmIHVuZGVyc2FtcGxlIGVsc2UgZGYuY29weSgpCiAgICBYLCBfID0gZmVhdHVyZV9tYXRyaXgoZGF0b3MpCgogICAgbW9kZWxvID0gS01lYW5zKG5fY2x1c3RlcnM9aywgaW5pdD0iay1tZWFucysrIiwgbl9pbml0PTEwLCByYW5kb21fc3RhdGU9cmFuZG9tX3N0YXRlKQogICAgZGF0b3NbImNsdXN0ZXIiXSA9IG1vZGVsby5maXRfcHJlZGljdChYKQogICAgcmV0dXJuIGRhdG9zLCBtb2RlbG8KCgpkZWYgcGNhXzJkKGRmOiBwZC5EYXRhRnJhbWUsICosIHJhbmRvbV9zdGF0ZTogaW50ID0gU0VNSUxMQSkgLT4gdHVwbGVbbnAubmRhcnJheSwgbnAubmRhcnJheV06CiAgICAiIiJQcm95ZWNjaW9uIGEgMkQgcGFyYSBncmFmaWNhciwgeSBsYSB2YXJpYW56YSBleHBsaWNhZGEgcG9yIGNhZGEgZWplLgoKICAgIFNlIGNhbGN1bGEgdW5hIHNvbGEgdmV6IHkgc2lydmUgcGFyYSB0b2RvcyBsb3MgazogbGEgcHJveWVjY2lvbiBubyBkZXBlbmRlCiAgICBkZWwgYWdydXBhbWllbnRvLgogICAgIiIiCiAgICBYLCBfID0gZmVhdHVyZV9tYXRyaXgoZGYpCiAgICBwY2EgPSBQQ0Eobl9jb21wb25lbnRzPTIsIHJhbmRvbV9zdGF0ZT1yYW5kb21fc3RhdGUpCiAgICByZXR1cm4gcGNhLmZpdF90cmFuc2Zvcm0oWCksIHBjYS5leHBsYWluZWRfdmFyaWFuY2VfcmF0aW9fCgoKZGVmIGNvbnRpbmdlbmN5KAogICAgZGY6IHBkLkRhdGFGcmFtZSwgKiwgY2x1c3Rlcjogc3RyID0gImNsdXN0ZXIiLCBldGlxdWV0YTogc3RyID0gImV2ZW50X2xhYmVsIgopIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIlRhYmxhIGNsdXN0ZXIgeCBldGlxdWV0YSByZWFsOiBjdWFudGFzIHZlbnRhbmFzIGRlIGNhZGEgdGlwbyBjYXllcm9uIGRvbmRlLiIiIgogICAgcmV0dXJuIHBkLmNyb3NzdGFiKGRmW2NsdXN0ZXJdLCBkZltldGlxdWV0YV0pCgoKZGVmIGxpZnQodGFibGE6IHBkLkRhdGFGcmFtZSkgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiQ3VhbnRvIHNlIGNvbmNlbnRyYSBjYWRhIGV0aXF1ZXRhIGVuIGNhZGEgY2x1c3RlciwgZnJlbnRlIGFsIGF6YXIuCgogICAgbGlmdCA9IFAoZXRpcXVldGEgfCBjbHVzdGVyKSAvIFAoZXRpcXVldGEpCgogICAgVW4gbGlmdCBkZSAzIHNpZ25pZmljYSBxdWUgZXNlIGNsdXN0ZXIgdGllbmUgZXNhIGNsYXNlIGRlIGV2ZW50byB0cmVzIHZlY2VzCiAgICBtYXMgY29uY2VudHJhZGEgZGUgbG8gcXVlIGxlIHRvY2FyaWEgcG9yIGF6YXIuIEVzIGxhIG1lZGlkYSBxdWUgZGljZSBzaSBlbAogICAgYWdydXBhbWllbnRvIHJlYWxtZW50ZSBlbmNvbnRybyBsb3MgZXZlbnRvcyBvIHNvbG8gcGFyZWNlIHF1ZSBzaTogdW4gbGlmdAogICAgY2VyY2FubyBhIDEgZW4gdG9kYSBsYSB0YWJsYSBzaWduaWZpY2EgcXVlIG5vIHNlcGFybyBuYWRhLgogICAgIiIiCiAgICBwX2RhZG9fY2x1c3RlciA9IHRhYmxhLmRpdih0YWJsYS5zdW0oYXhpcz0xKSwgYXhpcz0wKQogICAgcF9nbG9iYWwgPSB0YWJsYS5zdW0oYXhpcz0wKSAvIHRhYmxhLnRvX251bXB5KCkuc3VtKCkKICAgIHJldHVybiBwX2RhZG9fY2x1c3Rlci5kaXYocF9nbG9iYWwsIGF4aXM9MSkK",
}

for ruta, b64 in ARCHIVOS.items():
    p = pathlib.Path(ruta)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))

print(f"{len(ARCHIVOS)} modulos escritos")


## 4. Importar

In [ ]:
import gc
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

from tesis.data import cdf, catalogs
from tesis.features import windows, summary
from tesis.models import labeling, clustering as cl

# Las nueve entran por defecto. Se probo dejar fuera densidad y alpha ratio por
# su alta fraccion de ausentes, pero medirlo lo desmintio: a 80 min el AUC de
# ICME sube de 0,650 a 0,746. Un control (las 7 restringidas a las MISMAS
# ventanas que sobreviven con 9) dio 0,655, asi que el 95 % de la mejora viene
# de las variables y no del filtrado.
VARIABLES = ['Bx','By','Bz','Vx','Vy','Vz','Temperatura',
             'Densidad','Alpha to proton density']

# El subconjunto de siete, por si se quiere reproducir la comparacion:
VARIABLES_MINIMAS = ['Bx','By','Bz','Vx','Vy','Vz','Temperatura']

print(f'{len(VARIABLES)} variables: {VARIABLES}')

## 5. Cargar y cruzar

Tarda alrededor de un minuto.

In [ ]:
imf = cdf.load_many([f'{DATA}/IMF_2008.cdf', f'{DATA}/IMF_2015.cdf'], cdf.load_imf)
swe = cdf.load_many([f'{DATA}/SWE_2008.cdf', f'{DATA}/SWE_2015.cdf'], cdf.load_swe)
merged = cdf.merge_imf_swe(imf, cdf.resample_4min(swe))

icme = catalogs.load_icme(f'{DATA}/icme_list.csv')
sir  = catalogs.load_sir(f'{DATA}/STEREO_SIRs_extracted.csv')

print(f'{len(merged):,} puntos a 4 min')
print(f'{len(icme)} eventos ICME · {len(sir)} eventos SIR')

## 6. Ver cuántos faltantes tiene cada variable

Esta tabla es la que justifica qué variables entran al análisis.

In [ ]:
todas = ['Bx','By','Bz','Vx','Vy','Vz','Temperatura','Densidad','Alpha to proton density']
windows.nan_report(merged, todas)

## 7. La reducción de cada ventana

Este es el paso central del método y conviene verlo explícito, porque dentro de
`analizar()` queda escondido en una sola llamada.

Cada ventana de 20 puntos Ã— 9 variables se resume a **3 estadísticos por
variable**: mediana, desviación y tendencia. De 180 números a 27.

La tendencia es la pendiente de la recta de mínimos cuadrados. Como la abscisa
es siempre `0, 1, …, 19`, tiene fórmula cerrada y se calcula sobre todo el
tensor de una vez, en lugar de llamar a `scipy.stats.linregress` una vez por
ventana y variable. El resultado es idéntico (verificado contra `np.polyfit`)
pero mucho más rápido.


In [ ]:
# --- Antes de reducir ---
w_demo, s_demo = windows.make_windows(merged, VARIABLES, window_size=20, stride=200)
print('ANTES  ', w_demo.shape, ' = (ventanas, puntos, variables)')
print(f'         cada ventana son {w_demo.shape[1]} x {w_demo.shape[2]} = '
      f'{w_demo.shape[1]*w_demo.shape[2]} numeros')

# --- La reduccion ---
red_demo = summary.summarize_windows(windows.interpolate_windows(w_demo))
print('DESPUES', red_demo.shape, '    = (ventanas, features)')
print(f'         cada ventana queda en {red_demo.shape[1]} numeros '
      f'= {len(VARIABLES)} variables x 3 estadisticos')
print(f'         reduccion de {w_demo.shape[1]*w_demo.shape[2]/red_demo.shape[1]:.1f}x')

# --- Los nombres de las columnas ---
print('\nColumnas resultantes:')
print(summary.summary_columns(VARIABLES))

# --- Comprobacion: la mediana coincide con calcularla a mano? ---
wi = windows.interpolate_windows(w_demo)
manual = np.median(wi[0, :, 0])
marco  = summary.build_summary_frame(wi, s_demo, merged['Tiempo'], VARIABLES,
                                     window_size=20)
print(f"\nnp.median(ventana 0, Bx) = {manual:.6f}")
print(f"columna Bx_median[0]     = {marco['Bx_median'].iloc[0]:.6f}")
print('coinciden:', np.isclose(manual, marco['Bx_median'].iloc[0]))

# Y la tendencia contra polyfit
x = np.arange(20)
pend = np.polyfit(x, wi[0, :, 0], 1)[0]
print(f"\nnp.polyfit(ventana 0, Bx) = {pend:.6f}")
print(f"columna Bx_trend[0]       = {marco['Bx_trend'].iloc[0]:.6f}")
print('coinciden:', np.isclose(pend, marco['Bx_trend'].iloc[0]))

del w_demo, wi, red_demo
gc.collect()

El k-means **no** usa las variables originales: trabaja sobre esas
columnas resumidas, ya estandarizadas por año. Las ventanas crudas se liberan en
cuanto se calcula el resumen.

## 8. El análisis completo

**Cambia `WINDOW_SIZE` aquí.** Cada punto son 4 minutos:

| WINDOW_SIZE | Duración |
|---|---|
| 20 | 80 min |
| 90 | 6 h |
| 180 | 12 h |
| 360 | 24 h |
| 720 | 48 h |

### Sobre la memoria

Las ventanas largas son costosas. Con paso 1 se genera una ventana por cada
punto de la serie, así que una ventana de 24 h con 9 variables ocuparía
**6,8 GB**, más otra copia al interpolar. Ni Colab ni un portátil normal lo
aguantan.

`paso_seguro()` calcula automáticamente cada cuántos puntos empezar una ventana
nueva para mantenerse por debajo de `GB_MAX`. Si aun así te quedas sin memoria,
baja `GB_MAX`.

Además de ahorrar memoria es **estadísticamente preferible**: con paso 1 dos
ventanas consecutivas de 24 h comparten el 99,7 % de sus puntos, de modo que se
repite casi la misma ventana cientos de veces sin aportar información nueva.
Comprobado: a 24 h el resultado con paso 1 (`bal_acc` 0,5336) y con paso 16
(0,5236) es prácticamente el mismo.


In [ ]:
# ============================================================
#  LO QUE SE PUEDE TOCAR
# ============================================================
WINDOW_SIZE   = 360    # puntos por ventana. 20=80min, 90=6h, 180=12h, 360=24h
K             = 5      # numero de grupos
UNDERSAMPLE   = True   # equilibrar la clase Fondo antes de agrupar

MINIMO_VALIDO = 0.75   # fraccion minima de puntos buenos por variable
MODO_ANTIGUO  = False  # True = criterio de rachas + interpolacion (lo de antes)

GB_MAX        = 1.5    # techo de memoria para el tensor de ventanas
# ============================================================


def paso_seguro(window_size, n_vars, n_puntos=len(merged), gb_max=GB_MAX):
    """Elige el paso entre ventanas para no agotar la RAM.

    Con paso 1 se crea una ventana por cada punto: para 24 h eso son ~263.000
    ventanas de 360 puntos x 9 variables = 6,8 GB, mas otra copia al interpolar.
    Ni Colab ni un portatil normal lo aguantan.

    Y no es solo cuestion de memoria: con paso 1 dos ventanas consecutivas de
    24 h comparten el 99,7 % de sus puntos. Son casi la misma ventana repetida
    cientos de veces, lo que infla el numero de muestras sin anadir informacion
    e introduce autocorrelacion. Un paso mayor es ademas mas correcto.
    """
    bytes_por_ventana = window_size * n_vars * 8
    gb_paso1 = n_puntos * bytes_por_ventana / 1024**3
    return max(1, int(np.ceil(gb_paso1 / gb_max)))


def analizar(window_size, variables=VARIABLES, undersample=True, k=5, stride=None,
             minimo=MINIMO_VALIDO, modo_antiguo=MODO_ANTIGUO):
    if stride is None:
        stride = paso_seguro(window_size, len(variables))

    w, s = windows.make_windows(merged, variables,
                                window_size=window_size, stride=stride)
    n_gen = len(w)
    gb = w.nbytes / 1024**3

    if modo_antiguo:
        # Criterio original: descartar por rachas de NaN y despues interpolar
        w, s = windows.drop_gappy_windows(w, s, max_consec=1)
        w = windows.interpolate_windows(w)
        ignorar_nan = False
    else:
        # Criterio actual: exigir suficientes datos buenos y NO interpolar,
        # porque el descriptor es integral
        w, s = windows.drop_sparse_windows(w, s, minimo=minimo)
        ignorar_nan = True

    df = summary.build_summary_frame(w, s, merged['Tiempo'], variables,
                                     window_size=window_size,
                                     ignorar_nan=ignorar_nan)
    del w                    # el tensor ya no hace falta: se libera
    gc.collect()

    # Sin interpolar puede quedar alguna feature indefinida
    df = df.dropna().reset_index(drop=True)
    df = summary.standardize_by_year(df)
    df = labeling.label_windows(df, icme, sir, umbral=0.8)
    df = labeling.drop_frontera(df)

    datos, modelo = cl.run_kmeans(df, k=k, undersample=undersample)
    tab  = cl.contingency(datos)
    prob = tab.div(tab.sum(axis=1), axis=0)
    y    = datos['event_label'].values
    pred = datos['cluster'].map(prob.idxmax(axis=1)).values

    return {
        'df': df, 'datos': datos, 'tab': tab,
        'generadas': n_gen, 'stride': stride, 'GB': gb,
        'bal_acc': balanced_accuracy_score(y, pred),
        'auc_ICME': roc_auc_score((y=='ICME').astype(int), datos['cluster'].map(prob['ICME'])),
        'lift': cl.lift(tab),
    }


r = analizar(WINDOW_SIZE, undersample=UNDERSAMPLE, k=K)

print(f"criterio           : "
      f"{'rachas + interpolacion (ANTIGUO)' if MODO_ANTIGUO else f'>={MINIMO_VALIDO:.0%} valido, sin interpolar'}")
print(f"ventana            : {WINDOW_SIZE} puntos = {WINDOW_SIZE*4/60:.1f} h")
print(f"paso entre ventanas: {r['stride']}  (memoria usada: {r['GB']:.2f} GB)")
print(f"ventanas generadas : {r['generadas']:,}")
print(f"tras filtrar huecos: {len(r['df']):,}")
print(f"agrupadas          : {len(r['datos']):,}")
print(f"exactitud equilib. : {r['bal_acc']:.4f}   (azar = 0.333)")
print(f"AUC ICME           : {r['auc_ICME']:.4f}   (azar = 0.500)")
print(f"lift ICME maximo   : {r['lift']['ICME'].max():.3f}")

print('\nEtiquetas por anio:')
print(pd.crosstab(r['df']['start'].dt.year, r['df']['event_label']))

gc.collect()

## 9. Matriz de confusión y lift

In [ ]:
import seaborn as sns
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(r['tab'], annot=True, fmt='d', cmap='Blues', ax=a1)
a1.set_title(f'Ventanas por grupo · ventana {WINDOW_SIZE*4/60:.0f} h')
sns.heatmap(r['lift'], annot=True, fmt='.2f', cmap='RdYlGn', center=1.0, ax=a2)
a2.set_title('Lift (1.0 = azar)')
plt.tight_layout(); plt.show()

## 10. Barrido de la longitud de ventana

Repite el análisis para varios tamaños y compara. Es el experimento que
mostró que la ventana de 80 minutos era demasiado corta.

Tarda varios minutos: las ventanas grandes son costosas.

In [ ]:
TAMANOS = [20, 90, 180, 360]     # anade 720 (48 h) si quieres

filas = []
for ws in TAMANOS:
    try:
        x = analizar(ws)
    except MemoryError:
        print(f'ws={ws}: sin memoria. Baja GB_MAX y vuelve a intentar.')
        gc.collect()
        continue
    except Exception as e:
        print(f'ws={ws}: {e}')
        gc.collect()
        continue
    filas.append({'puntos': ws, 'horas': ws*4/60, 'paso': x['stride'],
                  'n': len(x['datos']),
                  'bal_acc': round(x['bal_acc'], 4),
                  'auc_ICME': round(x['auc_ICME'], 4),
                  'lift_ICME': round(float(x['lift']['ICME'].max()), 3)})
    print(f"ws={ws:4} ({ws*4/60:5.1f} h, paso {x['stride']:2})  "
          f"bal_acc={x['bal_acc']:.4f}  "
          f"AUC={x['auc_ICME']:.4f}  lift={x['lift']['ICME'].max():.2f}", flush=True)
    del x
    gc.collect()

barrido = pd.DataFrame(filas)
display(barrido)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(barrido['horas'], barrido['bal_acc'], 'o-', label='Exactitud equilibrada')
ax.plot(barrido['horas'], barrido['auc_ICME'], 's-', label='AUC ICME')
ax.axhline(1/3, ls='--', c='gray', label='azar (0.333)')
ax.set_xscale('log'); ax.set_xlabel('Ventana (horas)'); ax.set_ylabel('Métrica')
ax.set_title('Efecto de la longitud de ventana'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 11. Ventanas crudas frente a ventanas resumidas

Compara tres representaciones de **la misma ventana**, con todo lo demás igual:

| | Qué es | Dimensión |
|---|---|---|
| **A** | resumen: mediana, desviación y tendencia | 27 |
| **B** | cruda: la ventana entera aplanada | 180 |
| **C** | PCA sobre la cruda | 27 |

**Sobre las 27 componentes de C:** se eligen 27 porque es *exactamente la misma
dimensión que el resumen*, para que la comparación sea justa. No se eligieron
por un umbral de varianza — que expliquen el 95 % es el resultado, no el
criterio. Comparar 27 features contra 180 no sería limpio, porque k-means se
degrada al subir la dimensión.

**Por qué no basta con comentar la reducción:** el resto del código espera
columnas llamadas `Bx_median`, `Bx_std`, `Bx_trend`… `standardize_by_year` las
busca por ese sufijo, y `build_summary_frame` es quien además crea `start` y
`end`. Si se salta ese paso, no hay nada que estandarizar ni etiquetar. Hay que
montar el DataFrame a mano, que es lo que hace esta celda.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

WS_COMP = 20   # 80 min: la configuracion con mas muestras

w_c, s_c = windows.make_windows(merged, VARIABLES, window_size=WS_COMP, stride=1)
w_c, s_c = windows.drop_sparse_windows(w_c, s_c, minimo=MINIMO_VALIDO)
# Aqui si hay que interpolar: la variante "cruda" usa los 180 puntos uno a uno,
# y k-means no admite NaN. Es el precio de comparar representaciones.
w_c = windows.interpolate_windows(w_c)
print(f'{len(w_c):,} ventanas de {w_c.shape[1]} puntos x {w_c.shape[2]} variables')

tiempos = pd.to_datetime(merged['Tiempo']).to_numpy()
ini_c, fin_c = tiempos[s_c], tiempos[s_c + WS_COMP - 1]


def evaluar(X, nombre, columnas):
    """Mismo tratamiento para las tres representaciones."""
    df = pd.DataFrame(X, columns=columnas)
    df['start'], df['end'] = ini_c, fin_c
    df = summary.standardize_by_year(df, features=list(columnas))
    df = labeling.label_windows(df, icme, sir, umbral=0.8)
    df = labeling.drop_frontera(df)

    datos = cl.undersample_fondo(df).copy()
    modelo = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42)
    datos['cluster'] = modelo.fit_predict(datos[list(columnas)].to_numpy())

    tab  = cl.contingency(datos)
    prob = tab.div(tab.sum(axis=1), axis=0)
    y    = datos['event_label'].values
    pred = datos['cluster'].map(prob.idxmax(axis=1)).values

    r = {'representacion': nombre, 'dim': X.shape[1], 'n': len(datos),
         'bal_acc':  round(balanced_accuracy_score(y, pred), 4),
         'auc_ICME': round(roc_auc_score((y=='ICME').astype(int),
                                         datos['cluster'].map(prob['ICME'])), 4),
         'auc_SIR':  round(roc_auc_score((y=='SIR').astype(int),
                                         datos['cluster'].map(prob['SIR'])), 4),
         'lift_ICME': round(float(cl.lift(tab)['ICME'].max()), 3)}
    print(f"{nombre:16} dim={r['dim']:4}  bal={r['bal_acc']:.4f}  "
          f"AUC_ICME={r['auc_ICME']:.4f}  AUC_SIR={r['auc_SIR']:.4f}  "
          f"lift={r['lift_ICME']:.3f}", flush=True)
    return r


filas_rep = []

# A) resumen: 27 features
filas_rep.append(evaluar(summary.summarize_windows(w_c),
                         'A) resumen', summary.summary_columns(VARIABLES)))

# B) cruda: 180 features
X_cru = w_c.reshape(len(w_c), -1)
cols_cru = [f'p{t:02d}_{v}' for t in range(WS_COMP) for v in VARIABLES]
filas_rep.append(evaluar(X_cru, 'B) cruda', cols_cru))

# C) PCA a la MISMA dimension que el resumen
mu, sd = X_cru.mean(axis=0), X_cru.std(axis=0)
sd[sd == 0] = 1
pca = PCA(n_components=27, random_state=42)
X_pca = pca.fit_transform((X_cru - mu) / sd)
print(f'\nPCA(27) explica el {100*pca.explained_variance_ratio_.sum():.1f}% '
      f'de la varianza de las {X_cru.shape[1]} crudas')
# El sufijo _median es solo para que standardize_by_year reconozca las columnas.
filas_rep.append(evaluar(X_pca, 'C) PCA 27', [f'pc{i:02d}_median' for i in range(27)]))

del w_c, X_cru, X_pca
gc.collect()

display(pd.DataFrame(filas_rep))

## 12. Firmas físicas de cada grupo

Qué caracteriza físicamente a cada grupo, en unidades reales (nT, km/s, K,
cmâ»Â³) y no estandarizadas.

La clave es resumir **sin estandarizar** para poder leer los valores: la
estandarización solo hace falta para agrupar.

Para contrastar, estas son las firmas que la literatura asocia a cada estructura:

| | ICME | SIR |
|---|---|---|
| campo \|B\| | intenso | intenso en la compresión |
| variabilidad de B | **baja** (campo suave) | alta |
| Bz | rotante, a menudo sur | variable |
| temperatura | **anormalmente baja** | alta |
| densidad | normal o baja | **alta** |
| alpha/protón | **elevada** | normal |
| velocidad | variable | **gradiente positivo** |


In [ ]:
K_FIRMAS = 5

# Se resume SIN estandarizar para conservar las unidades fisicas
w_f, s_f = windows.make_windows(merged, VARIABLES, window_size=20, stride=1)
w_f, s_f = windows.drop_sparse_windows(w_f, s_f, minimo=MINIMO_VALIDO)

fisico = summary.build_summary_frame(w_f, s_f, merged['Tiempo'], VARIABLES,
                                     window_size=20, ignorar_nan=True)
del w_f
gc.collect()
fisico = fisico.dropna().reset_index(drop=True)

# El mismo marco, estandarizado y etiquetado, es el que se agrupa
escalado = summary.standardize_by_year(fisico)
escalado = labeling.label_windows(escalado, icme, sir, umbral=0.8)
escalado = labeling.drop_frontera(escalado)

# Se cruzan por 'start' porque el submuestreo reinicia el indice
fisico = fisico.merge(escalado[['start','event_label']], on='start',
                      how='inner', validate='one_to_one')

# Magnitudes derivadas para leer las firmas
fisico['|B|'] = np.sqrt(fisico['Bx_median']**2 + fisico['By_median']**2
                        + fisico['Bz_median']**2)
fisico['|V|'] = np.sqrt(fisico['Vx_median']**2 + fisico['Vy_median']**2
                        + fisico['Vz_median']**2)
fisico['B_varia'] = fisico[['Bx_std','By_std','Bz_std']].mean(axis=1)

COLUMNAS = [('|B|','|B| nT'), ('B_varia','var B'), ('Bz_median','Bz'),
            ('|V|','|V| km/s'), ('Vx_trend','tend Vx'),
            ('Densidad_median','dens'), ('Temperatura_median','temp K'),
            ('Alpha to proton density_median','alpha/p')]


def firmas(k):
    datos, _ = cl.run_kmeans(escalado, k=k, undersample=True)
    fis = fisico.merge(datos[['start','cluster']], on='start',
                       how='inner', validate='one_to_one')
    tab  = cl.contingency(datos)
    lift = cl.lift(tab)
    prop = tab.div(tab.sum(axis=1), axis=0)

    filas = []
    for c in sorted(fis['cluster'].unique()):
        sub = fis[fis['cluster'] == c]
        f = {'grupo': c, 'n': len(sub)}
        for col, etiq in COLUMNAS:
            f[etiq] = round(float(sub[col].median()), 3)
        f['%ICME'] = round(100*prop.loc[c,'ICME'], 1)
        f['lift_ICME'] = round(float(lift.loc[c,'ICME']), 2)
        f['lift_SIR']  = round(float(lift.loc[c,'SIR']), 2)
        filas.append(f)
    return pd.DataFrame(filas).sort_values('lift_ICME', ascending=False)


tabla_firmas = firmas(K_FIRMAS)
print(f'--- k = {K_FIRMAS} ---')
display(tabla_firmas)

print('\n--- referencia: viento de fondo ---')
fondo = fisico[fisico['event_label'] == 'Fondo']
for col, etiq in COLUMNAS:
    print(f'  {etiq:12} {fondo[col].median():12.3f}')

### Elegir k por las firmas, no por el codo

Si al subir `k` los grupos nuevos tienen la misma firma física que uno anterior
y solo difieren en una variable, se está subdividiendo sin describir una
estructura distinta. Eso da un criterio del dominio para fijar `k`, más
defendible que el método del codo o la silueta.

In [ ]:
for k in (5, 6, 7):
    t = firmas(k)
    print(f'\n{"="*92}\nk = {k}\n{"="*92}')
    print(t.to_string(index=False))
    gc.collect()

## 13. Resolución nativa: no promediar antes de resumir

Hasta aquí todo se promedia a 4 minutos para poder cruzar campo y plasma, y la
ventana son 20 de esos promedios. El coste no se ve: si dentro de esos 4 minutos
algo oscilaba, el promedio lo aplana **antes** de que lo midamos. La desviación
que sale es la de veinte promedios, no la de los datos.

Aquí la ventana se define por **duración** (80 min) y cada instrumento aporta sus
propios puntos, sin promediar nada.

Para cambiar el tamaño de ventana en esta variante se usa `DURACION`, no
`WINDOW_SIZE`: `'80min'`, `'6h'`, `'24h'`.


In [ ]:
from tesis.features import nativo

DURACION = '80min'   # <-- el tamano de ventana aqui va en tiempo
PASO     = '4min'    # cada cuanto se evalua una ventana nueva

VAR_MAG = ['Bx','By','Bz']
VAR_SWE = ['Vx','Vy','Vz','Temperatura','Densidad','Alpha to proton density']

# Cadencia real de cada instrumento, antes de tocar nada
for nombre, d in (('MAG', imf), ('SWEPAM', swe)):
    cad = pd.to_datetime(d['Tiempo']).diff().median()
    print(f'{nombre:8} {len(d):9,} puntos   cadencia {cad}')


def analizar_nativo(duracion=DURACION, paso=PASO, minimo=MINIMO_VALIDO, k=5,
                    undersample=False):
    df = nativo.resumir_nativo(
        {'mag': (imf, VAR_MAG), 'swe': (swe, VAR_SWE)},
        duracion=duracion, paso=paso, minimo_valido=minimo)
    df = summary.standardize_by_year(df)
    df = labeling.label_windows(df, icme, sir, umbral=0.8)
    df = labeling.drop_frontera(df)

    datos, modelo = cl.run_kmeans(df, k=k, undersample=undersample)
    tab  = cl.contingency(datos)
    prob = tab.div(tab.sum(axis=1), axis=0)
    y    = datos['event_label'].values
    pred = datos['cluster'].map(prob.idxmax(axis=1)).values
    return {'df': df, 'datos': datos, 'tab': tab, 'lift': cl.lift(tab),
            # por filas: de lo que cae en el grupo, que fraccion es de cada tipo
            'por_fila': (tab.div(tab.sum(axis=1), axis=0)*100).round(1),
            # por columnas: de todas las de ese tipo, cuantas cayeron aqui
            'por_columna': (tab.div(tab.sum(axis=0), axis=1)*100).round(1),
            'bal_acc': balanced_accuracy_score(y, pred),
            'auc_ICME': roc_auc_score((y=='ICME').astype(int),
                                      datos['cluster'].map(prob['ICME'])),
            'auc_SIR': roc_auc_score((y=='SIR').astype(int),
                                     datos['cluster'].map(prob['SIR']))}


# SUBMUESTREO: ponerlo en False es lo metodologicamente correcto para un
# metodo no supervisado. Equilibrar la clase Fondo exige mirar event_label, que
# sale del catalogo: si a priori no se sabe que es fondo, tampoco se puede
# equilibrar por esa columna. Medido sobre 2008 y 2015, con ventana de 80 min:
#
#                     % del grupo que es ICME   % de ICME capturadas   lift
#   con submuestreo          52,8 %                   27,6 %           5,20
#   sin submuestreo          13,7 %                   34,0 %           3,49
#
# La precision baja porque el 86 % de las ventanas son Fondo, asi que cualquier
# grupo estara dominado por Fondo. El lift es la medida que no se deja enganar
# por eso.
SUBMUESTREO = False

rn = analizar_nativo(undersample=SUBMUESTREO)
print(f"\nventana {DURACION}, paso {PASO}, >={MINIMO_VALIDO:.0%} valido, "
      f"submuestreo={SUBMUESTREO}")
print(f"  ventanas   : {len(rn['df']):,}")
print(f"  agrupadas  : {len(rn['datos']):,}")
print(f"  bal_acc    : {rn['bal_acc']:.4f}")
print(f"  AUC ICME   : {rn['auc_ICME']:.4f}")
print(f"  AUC SIR    : {rn['auc_SIR']:.4f}")
print(f"  lift ICME  : {rn['lift']['ICME'].max():.3f}")

# Las dos lecturas de la matriz, que no dicen lo mismo
print('\n--- de lo que cae en cada grupo, % de cada tipo (precision) ---')
display(rn['por_fila'])
print('--- de todas las de cada tipo, % que cayo en el grupo (exhaustividad) ---')
display(rn['por_columna'])

print('\nmejor grupo por clase:')
for clase in ('ICME', 'SIR'):
    g = int(rn['lift'][clase].idxmax())
    print(f"  {clase:5} -> grupo {g}: "
          f"{rn['por_fila'].loc[g, clase]:5.1f} % del grupo es {clase}  |  "
          f"{rn['por_columna'].loc[g, clase]:5.1f} % de las {clase} estan aqui  |  "
          f"lift {rn['lift'].loc[g, clase]:.2f}")
gc.collect()

Comparación directa contra el método que promedia a 4 minutos:

In [ ]:
ra = analizar(20)   # 20 puntos de 4 min = 80 min

print(f"{'metodo':26} {'ventanas':>9} {'bal_acc':>8} {'AUC_ICME':>9} {'lift':>7}")
print('-' * 64)
print(f"{'promediado a 4 min':26} {len(ra['df']):9,} {ra['bal_acc']:8.4f} "
      f"{ra['auc_ICME']:9.4f} {ra['lift']['ICME'].max():7.3f}")
print(f"{'resolucion nativa':26} {len(rn['df']):9,} {rn['bal_acc']:8.4f} "
      f"{rn['auc_ICME']:9.4f} {rn['lift']['ICME'].max():7.3f}")

## 14. Ver el método sobre un tramo concreto

Series temporales reales coloreadas por el grupo asignado. Arriba, los
intervalos del catálogo; debajo, las variables en su resolución y los
descriptores que realmente alimentan el agrupamiento.

Sirve para ver en casos concretos dónde acierta y dónde no, que es algo que
las métricas agregadas no muestran.

El grupo se pinta en una banda propia en vez de como fondo de cada panel: al
colorear el fondo, los grupos mayoritarios tapan todo y no se distingue nada.


In [ ]:
import matplotlib.dates as mdates
from matplotlib.patches import Patch

PALETA = ['#6355C4', '#009B8D', '#CB5A17', '#B0447E', '#8A8F9E']
DIAS = 12

# El modelo se entrena equilibrado pero se aplica a TODAS las ventanas
equilibrado, modelo = cl.run_kmeans(escalado, k=5, undersample=True)
X, _ = cl.feature_matrix(escalado)
todas = escalado.copy()
todas['cluster'] = modelo.predict(X)

tab  = cl.contingency(equilibrado)
lift = cl.lift(tab)
g_icme, g_sir = int(lift['ICME'].idxmax()), int(lift['SIR'].idxmax())
print(f'grupo ICME = {g_icme} (lift {lift.loc[g_icme,"ICME"]:.2f})')
print(f'grupo SIR  = {g_sir} (lift {lift.loc[g_sir,"SIR"]:.2f})')


def elegir_tramo(dias=DIAS):
    """Tramo con un ICME dentro, un SIR cerca y buena cobertura de datos."""
    esperadas = dias * 24 * 15
    mejor, mejor_cob = None, 0
    for _, ev in icme.iterrows():
        ini = ev['start_time'] - pd.Timedelta(days=dias*0.35)
        fin = ini + pd.Timedelta(days=dias)
        cob = ((todas['start'] >= ini) & (todas['start'] <= fin)).sum() / esperadas
        hay_sir = ((sir['start_time'] < fin) & (sir['end_time'] > ini)).any()
        if hay_sir and cob > mejor_cob:
            mejor, mejor_cob = (ini, fin), cob
    return mejor, mejor_cob


def dibujar_tramo(ini, fin):
    crudo = merged[(merged['Tiempo'] >= ini) & (merged['Tiempo'] <= fin)].copy()
    vent  = todas[(todas['start'] >= ini) & (todas['start'] <= fin)].sort_values('start')
    fis   = fisico.merge(vent[['start','cluster']], on='start', how='inner')

    crudo['|B|'] = np.sqrt(crudo['Bx']**2 + crudo['By']**2 + crudo['Bz']**2)
    crudo['|V|'] = np.sqrt(crudo['Vx']**2 + crudo['Vy']**2 + crudo['Vz']**2)

    # Las componentes van sin agrupar en modulo: la rotacion del campo es parte
    # de la firma de una ICME y el modulo la esconde.
    CRUDAS = [('Bx','Bx\n[nT]'), ('By','By\n[nT]'), ('Bz','Bz\n[nT]'),
              ('|B|','|B|\n[nT]'),
              ('Vx','Vx\n[km/s]'), ('Vy','Vy\n[km/s]'), ('Vz','Vz\n[km/s]'),
              ('Densidad','Np\n[cm$^{-3}$]'), ('Temperatura','Tp\n[K]'),
              ('Alpha to proton density','Na/Np')]
    ESTAD  = [('Bz_median','mediana\nBz'), ('Bz_std','desv.\nBz'),
              ('Temperatura_median','mediana\nTp'), ('Vx_trend','tendencia\nVx')]

    n = 1 + len(CRUDAS) + len(ESTAD)
    fig, ejes = plt.subplots(n, 1, figsize=(13, 1.35*n), sharex=True,
                             gridspec_kw={'height_ratios':[0.5]+[1]*(n-1),
                                          'hspace':0.1})

    ax = ejes[0]
    for _, r in vent.iterrows():
        ax.axvspan(r['start'], r['end'], ymin=0, ymax=0.55,
                   color=PALETA[int(r['cluster']) % len(PALETA)], lw=0)
    for cat, color, nom in ((icme,'#CB5A17','ICME'), (sir,'#009B8D','SIR')):
        sub = cat[(cat['start_time'] < fin) & (cat['end_time'] > ini)]
        for _, ev in sub.iterrows():
            ax.axvspan(max(ev['start_time'], ini), min(ev['end_time'], fin),
                       ymin=0.68, ymax=1.0, color=color, lw=0)
            ax.text(max(ev['start_time'], ini), 1.05, nom, fontsize=7, color=color,
                    transform=ax.get_xaxis_transform())
    ax.set_yticks([])
    ax.set_ylabel('catálogo\n—\ngrupo', fontsize=8, rotation=0, ha='right', va='center')
    for lado in ('top','right','left'):
        ax.spines[lado].set_visible(False)

    for ax, (col, etiq) in zip(ejes[1:], CRUDAS):
        ax.plot(crudo['Tiempo'], crudo[col], lw=0.6, color='#171A2B')
        ax.set_ylabel(etiq, fontsize=8, rotation=0, ha='right', va='center')
        ax.grid(alpha=0.2, lw=0.4); ax.tick_params(labelsize=7)

    for ax, (col, etiq) in zip(ejes[1+len(CRUDAS):], ESTAD):
        ax.plot(fis['start'], fis[col], lw=0.6, color='#B8BCC8', zorder=1)
        ax.scatter(fis['start'], fis[col], s=2.5, zorder=2, lw=0,
                   c=[PALETA[int(c) % len(PALETA)] for c in fis['cluster']])
        ax.set_ylabel(etiq, fontsize=8, rotation=0, ha='right', va='center')
        ax.grid(alpha=0.2, lw=0.4); ax.axhline(0, color='#AAA', lw=0.5, ls='--')
        ax.tick_params(labelsize=7)

    ejes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ejes[-1].set_xlim(ini, fin)

    leyenda = []
    for g in range(5):
        t = f'grupo {g}'
        if g == g_icme: t += ' (ICME)'
        if g == g_sir:  t += ' (SIR)'
        leyenda.append(Patch(facecolor=PALETA[g % len(PALETA)], label=t))
    fig.legend(handles=leyenda, loc='lower center', ncol=5, frameon=False,
               fontsize=8, bbox_to_anchor=(0.5, 0.055))
    fig.suptitle(f'{ini:%d %b %Y} – {fin:%d %b %Y}', fontsize=11, y=0.935)
    plt.show()


(ini, fin), cob = elegir_tramo()
print(f'tramo: {ini:%Y-%m-%d} a {fin:%Y-%m-%d}   cobertura {cob:.0%}')
dibujar_tramo(ini, fin)

Para mirar otro periodo, llama a `dibujar_tramo` con las fechas que
quieras:

```python
dibujar_tramo(pd.Timestamp('2008-03-01'), pd.Timestamp('2008-03-20'))
```

## 15. Comparar juegos de variables

Para responder a la pregunta de si conviene incluir densidad y alpha ratio
pese a sus faltantes.

In [ ]:
SIETE = ['Bx','By','Bz','Vx','Vy','Vz','Temperatura']
NUEVE = SIETE + ['Densidad','Alpha to proton density']

for nombre, vs in [('7 variables', SIETE), ('9 variables', NUEVE)]:
    x = analizar(WINDOW_SIZE, variables=vs)
    print(f"{nombre:14} generadas={x['generadas']:,}  agrupadas={len(x['datos']):,}  "
          f"bal_acc={x['bal_acc']:.4f}  AUC={x['auc_ICME']:.4f}  "
          f"lift={x['lift']['ICME'].max():.2f}")